<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T05-L03-gptq-and-awq-from-the-math/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T05-L03-gptq-and-awq-from-the-math/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T05-L03-gptq-and-awq-from-the-math/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T05-L03-gptq-and-awq-from-the-math/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T05-L03 · GPTQ and AWQ from the maths: error-compensating rounding and activation-aware scaling

**You will build:** two widely used post-training quantisation methods for language models,
from the mathematics up and in numpy alone. **GPTQ** rounds a weight
matrix one input channel at a time and pushes each rounding error onto the weights that are not
rounded yet. **AWQ** scales up the input channels that carry large activations before it
rounds. You will measure both against plain round-to-nearest, at 4 and 3 bits, on a small
language model that this notebook trains for itself.

**Time:** ~2.5 hours · **Runs on:** a laptop CPU, no GPU, no download, no network ·
**Prerequisites:** T03-L03-nanolm-cpu-slice.

By the end you will be able to:

1. Implement a group-wise integer grid and round-to-nearest (RTN) quantisation, and measure
   the error it adds to a layer's output at 4 and 3 bits.
2. Derive the layer's output error as a quadratic form in the weight error with the Hessian
   XᵀX / n, and implement the Optimal Brain Surgeon update that compensates one rounding error,
   checked against a brute-force constrained minimiser.
3. Implement GPTQ (a damped inverse-Hessian Cholesky factor and column-by-column rounding with
   compensation), prove it equals RTN when the Hessian is diagonal and equals explicit
   elimination otherwise, and measure that it raises the weight error while lowering the output error.
4. Implement AWQ's activation-aware scales and their grid search, and measure which structure
   in a layer's inputs (unequal magnitudes, or correlated channels) each method can use.
5. Measure output error, KL divergence and held-out loss at 4 and 3 bits for all three methods,
   and the calibration-set size at which each stops improving.

**This is not a real LLM.** The model has a few tens of thousands of quantised weights, and the
lesson says where that makes the numbers differ from a billion-parameter model. This notebook
carries a compact copy of the T03-L03 model, trains its own checkpoint at the start, and
never loads another lesson's files.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T03-L03`

**This lesson assumes you can already:**

- multiply matrices with @ in numpy, take a matrix inverse and a trace, and predict the shape of each result
- say what a Hessian is, and why the Hessian of a squared-error loss in the weights does not depend on the weights
- read the attention and residual lines of the small transformer block in T03-L03, and a two-layer ReLU MLP, without being told what each does
- round a number to the nearest multiple of a step and say how large the error can be

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers](https://arxiv.org/abs/2210.17323) · Frantar, Ashkboos, Hoefler and Alistarh (ICLR 2023, arXiv) · *free* — section 3 is the layer-wise objective and the Optimal Brain Quantization update, and section 4 is the fixed order and the Cholesky reformulation that exercise 5 implements (and the lazy batches it does not)
- [AWQ: Activation-aware Weight Quantization for LLM Compression and Acceleration](https://arxiv.org/abs/2306.00978) · Lin et al. (MLSys 2024, arXiv) · *free* — section 3.2 derives why scaling up a channel with large activations shrinks its relative quantisation error (equation 2) and sets out the alpha grid search that exercises 6 and 7 implement
- [Optimal Brain Compression: A Framework for Accurate Post-Training Quantization and Pruning](https://arxiv.org/abs/2208.11580) · Frantar, Singh and Alistarh (NeurIPS 2022, arXiv) · *free* — an exact and efficient realisation of Optimal Brain Surgeon for pruning and quantisation, the starting point GPTQ builds on and speeds up, and where the layer-wise reconstruction problem is set out
- [Mathematics for Machine Learning, section 5.7 (Higher-Order Derivatives) and section 4.3 (Cholesky Decomposition)](https://mml-book.github.io/) · Deisenroth, Faisal and Ong (Cambridge University Press; PDF free from the authors) · *free* — 5.7 defines the Hessian as the matrix of second derivatives, constant for a squared error in the weights (exercise 3); 4.3 gives the factorisation H = L Lt that exercise 5 reads GPTQ's updates from

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import os
import signal
import sys
import threading
import time
import traceback
from typing import Callable

import numpy as np

print("python", sys.version.split()[0], "· numpy", np.__version__)

INIT_SEED = 1234            # the checkpoint's initial weights
TRAIN_SEED = 0              # the order of the training batches
SYNTH_SEED = 20261030       # every synthetic layer in this notebook derives from this
GROUP_SIZE = 16             # input channels that share one scale and one zero point
PERCDAMP = 0.01             # GPTQ dampening, as a fraction of the mean Hessian diagonal
N_GRID = 20                 # AWQ tries alpha = j / N_GRID for j = 0 .. N_GRID - 1
CALIB_TOKENS = 2048         # calibration-set size for the main comparison
SWEEP_TOKENS = (16, 32, 64, 128, 256, 512, 1024, 2048, 4096)
CALIB_DRAWS = (0, 1, 2)     # independent calibration draws per size
KNEE_TOL = 0.05             # "stopped improving" = within 5% of the largest-set error
BITS = (4, 3)
_CHECK_SECONDS = 2        # time limit for each exercise check: they work on tiny inputs and take a few milliseconds
_DEMO_SECONDS = 15

_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("find_params", "quantize_to_grid"),
    "exercise 2": ("rtn_quantize",),
    "exercise 3": ("layer_hessian", "quad_form_error"),
    "exercise 4": ("obs_update",),
    "exercise 5": ("damped_inverse_factor", "gptq_quantize"),
    "exercise 6": ("awq_scales",),
    "exercise 7": ("awq_quantize",),
    "exercise 8": ("knee_size",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (layer_hessian, quad_form_error)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


class _Timeout(BaseException):
    """Raised by `_watchdog`. A BaseException, so a student's `except Exception` cannot swallow it."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _watchdog(seconds: float, memory_mib: float = 2048):
    """Stop a block that runs longer than `seconds`, so a loop that never ends becomes a message
    instead of a frozen notebook. It uses SIGALRM, which exists on Linux and macOS (Colab, Kaggle,
    Binder, Codespaces and most laptops) and only in the main thread; where it does not exist the
    block simply runs unguarded. An alarm that was already running (the grader's own time limit)
    keeps its deadline if it is due first and is put back, with the time it had left, afterwards."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_watchdog, "stopped", False):           # a stopped block's frames keep what they built in a
        _watchdog.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = f"still running after {seconds:g} s"
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _Timeout(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _Timeout(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _Timeout(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_watchdog", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _Timeout as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _watchdog.stopped = True
        raise _Timeout(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _watchdog.stopped = True
        raise _Timeout(state["message"])


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = (),
         seconds: float = 0) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A call that is still running after
    `seconds` (a loop that never ends, or a very slow implementation) is stopped and reported. A demo names the exercises it `needs`: until each has passed its
    check, the demo says which one it is waiting for and skips. Nothing is swallowed: every
    outcome is recorded in `_STATUS` for the progress board at the foot of the notebook, and
    every failure in `_FAILED_CHECKS`, which ends a script run non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        with _watchdog(seconds or (_CHECK_SECONDS if label in _EXERCISES else _DEMO_SECONDS)):
            check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except _Timeout as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}. Either a loop that never ends (check that every loop advances or removes "
              "something each time round) or an implementation too slow for this input.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. The model you are quantising

This is the character-level model of T03-L03, carried over so the notebook stands alone: token
and position embeddings, one causal attention head, a residual connection and a linear head,
trained on an excerpt of Shakespeare's Sonnets (public domain, see `claims.yaml`). It has one
addition. T03-L03's closing section lists the missing **MLP sub-layer**, so this copy has one: a linear layer up to `HIDDEN` units, a ReLU, and
a linear layer back, added to the residual stream. Its backward pass sends the gradient back through the second
linear layer, through the ReLU (only where its input was positive) and through the first linear layer.

Six weight matrices will be quantised: the query, key, value and output projections of
attention (`Wq`, `Wk`, `Wv`, `Wo`) and the two MLP matrices (`Wf1`, `Wf2`). Embeddings, biases and
the final linear head stay in full precision here. Each matrix is stored as
`(inputs, outputs)` and applied as `y = x @ W`, so a **row** of `W` is one input channel.

Read the code, then run the cell after it: it trains the checkpoint and
prints how it did against the unigram baseline (a model that knows only how often each character occurs) and on its own
training windows.

In [ ]:
CORPUS = """From fairest creatures we desire increase,
That thereby beauty's rose might never die,
But as the riper should by time decease,
His tender heir might bear his memory:
But thou, contracted to thine own bright eyes,
Feed'st thy light's flame with self-substantial fuel,
Making a famine where abundance lies,
Thyself thy foe, to thy sweet self too cruel:
Thou that art now the world's fresh ornament,
And only herald to the gaudy spring,
Within thine own bud buriest thy content,
And tender churl mak'st waste in niggarding:
    Pity the world, or else this glutton be,
    To eat the world's due, by the grave and thee.

II

When forty winters shall besiege thy brow,
And dig deep trenches in thy beauty's field,
Thy youth's proud livery so gazed on now,
Will be a tatter'd weed of small worth held:
Then being asked, where all thy beauty lies,
Where all the treasure of thy lusty days;
To say, within thine own deep sunken eyes,
Were an all-eating shame, and thriftless praise.
How much more praise deserv'd thy beauty's use,
If thou couldst answer 'This fair child of mine
Shall sum my count, and make my old excuse,'
Proving his beauty by succession thine!
    This were to be new made when thou art old,
    And see thy blood warm when thou feel'st it cold.

III

Look in thy glass and tell the face thou viewest
Now is the time that face should form another;
Whose fresh repair if now thou not renewest,
Thou dost beguile the world, unbless some mother.
For where is she so fair whose unear'd womb
Disdains the tillage of thy husbandry?
Or who is he so fond will be the tomb,
Of his self-love to stop posterity?
Thou art thy mother's glass and she in thee
Calls back the lovely April of her prime;
So thou through windows of thine age shalt see,
Despite of wrinkles this thy golden time.
    But if thou live, remember'd not to be,
    Die single and thine image dies with thee.

IV

Unthrifty loveliness, why dost thou spend
Upon thyself thy beauty's legacy?
Nature's bequest gives nothing, but doth lend,
And being frank she lends to those are free:
Then, beauteous niggard, why dost thou abuse
The bounteous largess given thee to give?
Profitless usurer, why dost thou use
So great a sum of sums, yet canst not live?
For having traffic with thyself alone,
Thou of thyself thy sweet self dost deceive:
Then how when nature calls thee to be gone,
What acceptable audit canst thou leave?
    Thy unused beauty must be tombed with thee,
    Which, used, lives th' executor to be.

V

Those hours, that with gentle work did frame
The lovely gaze where every eye doth dwell,
Will play the tyrants to the very same
And that unfair which fairly doth excel;
For never-resting time leads summer on
To hideous winter, and confounds him there;
Sap checked with frost, and lusty leaves quite gone,
Beauty o'er-snowed and bareness every where:
Then were not summer's distillation left,
A liquid prisoner pent in walls of glass,
Beauty's effect with beauty were bereft,
Nor it, nor no remembrance what it was:
    But flowers distill'd, though they with winter meet,
    Leese but their show; their substance still lives sweet.


VI

Then let not winter's ragged hand deface,
In thee thy summer, ere thou be distill'd:
Make sweet some vial; treasure thou some place
With beauty's treasure ere it be self-kill'd.
That use is not forbidden usury,
Which happies those that pay the willing loan;
That's for thyself to breed another thee,
Or ten times happier, be it ten for one;
Ten times thyself were happier than thou art,
If ten of thine ten times refigur'd thee:
Then what could death do if thou shouldst depart,
Leaving thee living in posterity?
    Be not self-will'd, for thou art much too fair
    To be death's conquest and make worms thine heir.

VII

Lo! in the orient when the gracious light
Lifts up his burning head, each under eye
Doth homage to his new-appearing sight,
Serving with looks his sacred majesty;
And having climb'd the steep-up heavenly hill,
Resembling strong youth in his middle age,
Yet mortal looks adore his beauty still,
Attending on his golden pilgrimage:
But when from highmost pitch, with weary car,
Like feeble age, he reeleth from the day,
The eyes, 'fore duteous, now converted are
From his low tract, and look another way:
    So thou, thyself outgoing in thy noon:
    Unlook'd, on diest unless thou get a son.

VIII

Music to hear, why hear'st thou music sadly?
Sweets with sweets war not, joy delights in joy:
Why lov'st thou that which thou receiv'st not gladly,
Or else receiv'st with pleasure thine annoy?
If the true concord of well-tuned sounds,
By unions married, do offend thine ear,
They do but sweetly chide thee, who confounds
In singleness the parts that thou shouldst bear.
Mark how one string, sweet husband to another,
Strikes each in each by mutual ordering;
Resembling sire and child and happy mother,
Who, all in one, one pleasing note do sing:
    Whose speechless song being many, seeming one,
    Sings this to thee: 'Thou single wilt prove none.'

IX

Is it for fear to wet a widow's eye,
That thou consum'st thyself in single life?
Ah! if thou issueless shalt hap to die,
The world will wail thee like a makeless wife;
The world will be thy widow and still weep
That thou no form of thee hast left behind,
When every private widow well may keep
By children's eyes, her husband's shape in mind:
Look! what an unthrift in the world doth spend
Shifts but his place, for still the world enjoys it;
But beauty's waste hath in the world an end,
And kept unused the user so destroys it.
    No love toward others in that bosom sits
    That on himself such murd'rous shame commits.

X

For shame! deny that thou bear'st love to any,
Who for thyself art so unprovident.
Grant, if thou wilt, thou art belov'd of many,
But that thou none lov'st is most evident:
For thou art so possess'd with murderous hate,
That 'gainst thyself thou stick'st not to conspire,
Seeking that beauteous roof to ruinate
Which to repair should be thy chief desire.
"""

VOCAB = sorted(set(CORPUS))
VOCAB_SIZE = len(VOCAB)
STOI = {ch: i for i, ch in enumerate(VOCAB)}


def encode(text: str) -> list[int]:
    return [STOI[ch] for ch in text]


DATA = np.array(encode(CORPUS), dtype=np.int64)
_SPLIT = int(0.9 * len(DATA))
TRAIN_DATA, VAL_DATA = DATA[:_SPLIT], DATA[_SPLIT:]

D_MODEL = 48        # width of the residual stream
HIDDEN = 192        # width of the MLP's hidden layer
CONTEXT = 64        # positions the model has an embedding for; also the window length used below

# The six matrices that get quantised, and which activation each one multiplies.
LAYER_INPUT = {"Wq": "x", "Wk": "x", "Wv": "x", "Wo": "ctx", "Wf1": "h1", "Wf2": "m"}
QUANTISED = tuple(LAYER_INPUT)


def softmax(x: np.ndarray) -> np.ndarray:
    """Softmax over the last axis, shifted by the row maximum so large scores cannot overflow."""
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def init_params(seed: int = INIT_SEED, scale: float = 0.02) -> dict:
    """Fresh float64 parameters from a seeded generator."""
    rng = np.random.default_rng(seed)
    d, h = D_MODEL, HIDDEN
    return {"E": rng.normal(0, scale, (VOCAB_SIZE, d)), "P": rng.normal(0, scale, (CONTEXT, d)),
            "Wq": rng.normal(0, scale, (d, d)), "Wk": rng.normal(0, scale, (d, d)),
            "Wv": rng.normal(0, scale, (d, d)), "Wo": rng.normal(0, scale, (d, d)),
            "Wf1": rng.normal(0, scale, (d, h)), "bf1": np.zeros(h),
            "Wf2": rng.normal(0, scale, (h, d)), "bf2": np.zeros(d),
            "W1": rng.normal(0, scale, (d, VOCAB_SIZE)), "b1": np.zeros(VOCAB_SIZE)}


def model_forward(params: dict, inputs: np.ndarray, targets=None):
    """Whole-sequence forward pass, (B, T) token ids in. Returns (loss, cache); loss is None without
    targets. The cache holds every intermediate: the activations that feed each quantised matrix are
    `x` (Wq, Wk, Wv), `ctx` (Wo), `h1` (Wf1) and `m` (Wf2)."""
    b, t = inputs.shape
    if t > CONTEXT:
        raise ValueError(f"{t} positions, but the model has only {CONTEXT} position embeddings")
    x = params["E"][inputs] + params["P"][None, :t]
    q, k, v = x @ params["Wq"], x @ params["Wk"], x @ params["Wv"]
    scores = (q @ k.transpose(0, 2, 1)) / np.sqrt(D_MODEL)
    scores = np.where(np.tril(np.ones((t, t), dtype=bool))[None], scores, -np.inf)
    a = softmax(scores)
    ctx = a @ v
    h1 = x + ctx @ params["Wo"]
    z = h1 @ params["Wf1"] + params["bf1"]
    m = np.maximum(z, 0.0)
    h = h1 + m @ params["Wf2"] + params["bf2"]
    logits = h @ params["W1"] + params["b1"]
    loss = None
    if targets is not None:
        shifted = logits - logits.max(axis=-1, keepdims=True)
        logp = shifted - np.log(np.exp(shifted).sum(axis=-1, keepdims=True))
        loss = float(-np.take_along_axis(logp, targets[..., None], axis=-1).mean())
    return loss, {"inputs": inputs, "targets": targets, "x": x, "q": q, "k": k, "v": v, "a": a,
                  "ctx": ctx, "h1": h1, "z": z, "m": m, "h": h, "logits": logits}


def model_backward(params: dict, cache: dict) -> dict:
    """Gradients of the mean cross-entropy with respect to every parameter (T03-L03 sections 8-10,
    plus the MLP: the ReLU passes gradient only where its input was positive)."""
    b, t = cache["inputs"].shape
    d, hid = D_MODEL, HIDDEN
    onehot = np.zeros_like(cache["logits"])
    np.put_along_axis(onehot, cache["targets"][..., None], 1.0, axis=-1)
    d_logits = (softmax(cache["logits"]) - onehot) / (b * t)
    grads = {"W1": cache["h"].reshape(-1, d).T @ d_logits.reshape(-1, VOCAB_SIZE),
             "b1": d_logits.sum(axis=(0, 1))}
    d_h = d_logits @ params["W1"].T
    grads["Wf2"] = cache["m"].reshape(-1, hid).T @ d_h.reshape(-1, d)
    grads["bf2"] = d_h.sum(axis=(0, 1))
    d_z = (d_h @ params["Wf2"].T) * (cache["z"] > 0)
    grads["Wf1"] = cache["h1"].reshape(-1, d).T @ d_z.reshape(-1, hid)
    grads["bf1"] = d_z.sum(axis=(0, 1))
    d_h1 = d_h + d_z @ params["Wf1"].T
    grads["Wo"] = cache["ctx"].reshape(-1, d).T @ d_h1.reshape(-1, d)
    d_ctx = d_h1 @ params["Wo"].T
    a, q, k, v, x = cache["a"], cache["q"], cache["k"], cache["v"], cache["x"]
    d_a = d_ctx @ v.transpose(0, 2, 1)
    d_v = a.transpose(0, 2, 1) @ d_ctx
    d_scores = a * (d_a - (d_a * a).sum(axis=-1, keepdims=True)) / np.sqrt(d)
    d_q, d_k = d_scores @ k, d_scores.transpose(0, 2, 1) @ q
    xf = x.reshape(-1, d)
    grads["Wq"], grads["Wk"], grads["Wv"] = (xf.T @ d_q.reshape(-1, d), xf.T @ d_k.reshape(-1, d),
                                             xf.T @ d_v.reshape(-1, d))
    d_x = d_h1 + d_q @ params["Wq"].T + d_k @ params["Wk"].T + d_v @ params["Wv"].T
    grads["P"] = np.zeros_like(params["P"])
    grads["P"][:t] = d_x.sum(axis=0)
    grads["E"] = np.zeros_like(params["E"])
    np.add.at(grads["E"], cache["inputs"], d_x)
    return grads


def make_batch(data: np.ndarray, batch_size: int, context_len: int, rng):
    """`batch_size` random (input, target) windows; the target is the input shifted by one."""
    starts = rng.integers(0, len(data) - context_len, size=batch_size)
    return (np.stack([data[s:s + context_len] for s in starts]),
            np.stack([data[s + 1:s + context_len + 1] for s in starts]))


def train_checkpoint(steps: int = 1500, lr: float = 3e-3, seed: int = TRAIN_SEED, batch_size: int = 32):
    """Train from INIT_SEED with Adam (T03-L03 section 13). Returns (params, [(step, loss)])."""
    params = init_params()
    m = {name: np.zeros_like(p) for name, p in params.items()}
    s = {name: np.zeros_like(p) for name, p in params.items()}
    rng = np.random.default_rng(seed)
    history = []
    for step in range(1, steps + 1):
        inputs, targets = make_batch(TRAIN_DATA, batch_size, CONTEXT, rng)
        loss, cache = model_forward(params, inputs, targets)
        grads = model_backward(params, cache)
        for name, p in params.items():
            m[name] = 0.9 * m[name] + 0.1 * grads[name]
            s[name] = 0.999 * s[name] + 0.001 * grads[name] ** 2
            p -= lr * (m[name] / (1 - 0.9 ** step)) / (np.sqrt(s[name] / (1 - 0.999 ** step)) + 1e-8)
        if step == 1 or step % 500 == 0:
            history.append((step, loss))
    return params, history


def windows(data: np.ndarray, length: int = CONTEXT):
    """Consecutive non-overlapping (input, target) windows covering `data`, as two (N, length) arrays."""
    n = (len(data) - 1) // length
    return (np.stack([data[i * length:i * length + length] for i in range(n)]),
            np.stack([data[i * length + 1:i * length + length + 1] for i in range(n)]))


TRAIN_X, TRAIN_Y = windows(TRAIN_DATA)
VAL_X, VAL_Y = windows(VAL_DATA)

_counts = np.bincount(TRAIN_DATA, minlength=VOCAB_SIZE).astype(np.float64)
_p = _counts[_counts > 0] / _counts.sum()
UNIGRAM_ENTROPY = float(-(_p * np.log(_p)).sum())
print(f"corpus: {len(CORPUS):,} characters, {VOCAB_SIZE} distinct; training windows {TRAIN_X.shape[0]}, "
      f"held-out windows {VAL_X.shape[0]}, each {CONTEXT} characters")
_all = init_params()
_n_q = sum(int(_all[n].size) for n in QUANTISED)
print(f"the model has {sum(int(a.size) for a in _all.values()):,} parameters, of which {_n_q:,} sit in the "
      f"{len(QUANTISED)} matrices that will be quantised ({', '.join(f'{n} {_all[n].shape[0]}x{_all[n].shape[1]}' for n in QUANTISED)})")

In [ ]:
PARAMS: dict = {}


def _train_and_report() -> None:
    params, history = train_checkpoint()
    for step, loss in history:
        print(f"  step {step:5d}  training loss {loss:.4f}")
    PARAMS.update(params)
    held = model_forward(params, VAL_X, VAL_Y)[0]
    train = model_forward(params, TRAIN_X, TRAIN_Y)[0]
    print(f"loss in nats per character, on the held-out windows {held:.3f}, on the training windows {train:.3f}; "
          f"the unigram baseline is {UNIGRAM_ENTROPY:.3f}")
    print(f"the model {'beats' if held < UNIGRAM_ENTROPY else 'does not beat'} the unigram baseline on text it never saw, "
          f"and its training loss is {'below' if train < held else 'not below'} its held-out loss: "
          f"{'it has partly memorised the excerpt' if train < held else 'no memorisation yet'}")


if __name__ == "__main__":
    _train_and_report()

### How the lesson measures damage

Three measurements recur, all computed by the given code below. For one matrix, the **layer output
error** is the quantity quantisation methods try to keep small: how far `X @ W_hat` is from
`X @ W` on a set of inputs `X`, summed over the output channels and averaged over the rows of `X`
(`output_error`). Divided by the energy of the true output it becomes a **relative** error
(`rel_output_error`), printed as a percentage. For the whole model, **KL divergence** measures how
far the quantised model's next-character distribution is from the full-precision model's, in
nats per position (it is never negative), and **held-out loss** is the ordinary cross-entropy.

Each matrix is quantised from the inputs the *full-precision* model sends it, one matrix at a time; the
last section says what a deeper model would do differently. Those inputs come from a **calibration set**
of training text, and the errors are always measured on the held-out text.

In [ ]:
def layer_inputs(params: dict, inputs: np.ndarray) -> dict:
    """The activation each quantised matrix multiplies, at every position of `inputs`: a dict from
    the matrix name to an (n_positions, n_input_channels) array."""
    _, cache = model_forward(params, inputs)
    flat = {"x": cache["x"], "ctx": cache["ctx"], "h1": cache["h1"], "m": cache["m"]}
    return {name: flat[key].reshape(-1, flat[key].shape[-1]) for name, key in LAYER_INPUT.items()}


def output_error(X: np.ndarray, W: np.ndarray, W_hat: np.ndarray) -> float:
    """Mean over the rows of X of the squared length of (x W - x W_hat): the layer-wise objective."""
    return float(np.mean(np.sum((X @ (W - W_hat)) ** 2, axis=1)))


def rel_output_error(X: np.ndarray, W: np.ndarray, W_hat: np.ndarray) -> float:
    """output_error divided by the mean squared length of the true output x W."""
    return output_error(X, W, W_hat) / float(np.mean(np.sum((X @ W) ** 2, axis=1)))


def weight_error(W: np.ndarray, W_hat: np.ndarray) -> float:
    """Sum of squared differences between the weights themselves: the error RTN minimises."""
    return float(np.sum((W - W_hat) ** 2))


def bits_per_weight(bits: int, group_size: int, scale_bits: int = 16) -> float:
    """Storage per weight: the integer, plus one `scale_bits`-bit scale and one `bits`-bit zero point
    shared by the `group_size` weights of a group."""
    return bits + (scale_bits + bits) / group_size


def calibration_set(pool: dict, n_tokens: int, draw: int) -> dict:
    """`n_tokens` positions drawn without replacement from `pool` (a dict from matrix name to its
    activations), the same positions for every matrix. Draws are nested: for one `draw`, a smaller set
    is a prefix of a larger one."""
    n_all = next(iter(pool.values())).shape[0]
    chosen = np.random.default_rng(1000 + draw).permutation(n_all)[:n_tokens]
    return {name: X[chosen] for name, X in pool.items()}


def log_probs(params: dict, inputs: np.ndarray) -> np.ndarray:
    logits = model_forward(params, inputs)[1]["logits"]
    shifted = logits - logits.max(axis=-1, keepdims=True)
    return shifted - np.log(np.exp(shifted).sum(axis=-1, keepdims=True))


def kl_to_full_precision(reference: dict, params: dict, inputs: np.ndarray) -> float:
    """Mean over positions of KL(p_reference || p_params), in nats."""
    lp_ref, lp = log_probs(reference, inputs), log_probs(params, inputs)
    return float(np.mean(np.sum(np.exp(lp_ref) * (lp_ref - lp), axis=-1)))


def with_weights(params: dict, replacements: dict) -> dict:
    """A copy of `params` with some matrices replaced."""
    out = dict(params)
    out.update(replacements)
    return out


def _percent(x: float) -> str:
    return f"{100 * x:.3f}%"


def _print_table(header: list, rows: list, widths: list = None) -> None:
    """A plain-text table: header and rows are lists of strings."""
    widths = widths or [max(len(str(r[i])) for r in [header] + rows) for i in range(len(header))]
    print("  ".join(str(h).ljust(w) if i == 0 else str(h).rjust(w) for i, (h, w) in enumerate(zip(header, widths))))
    for r in rows:
        print("  ".join(str(c).ljust(w) if i == 0 else str(c).rjust(w) for i, (c, w) in enumerate(zip(r, widths))))


POOL: dict = {}        # calibration pool: the activations of the training windows
HELD: dict = {}        # the activations of the held-out windows
BASELINE: dict = {}    # full-precision held-out numbers


def _build_pools() -> None:
    POOL.update(layer_inputs(PARAMS, TRAIN_X))
    HELD.update(layer_inputs(PARAMS, VAL_X))
    BASELINE["loss"] = model_forward(PARAMS, VAL_X, VAL_Y)[0]
    n_pool = next(iter(POOL.values())).shape[0]
    print(f"calibration pool: {n_pool:,} training positions; held-out set: "
          f"{next(iter(HELD.values())).shape[0]:,} positions; full-precision held-out loss "
          f"{BASELINE['loss']:.3f} nats per character")


if __name__ == "__main__":
    _build_pools()

## 2. Exercise 1: the integer grid

Storing a weight in `b` bits means choosing one of `2**b` evenly spaced values. The grid is
described by a **scale** (the spacing) and a **zero point** (which integer stands for 0.0), and
each group of input channels gets its own pair for every output channel, so one outlier weight
only coarsens the grid of the weights it shares a group with. The lesson uses the usual
asymmetric min-max grid, with one change that keeps pruned weights honest: the range is widened
to include 0, so that an exact zero is always stored exactly.

For a block `w` of shape `(g, d_out)` (the `g` rows are the group's input channels), for each
output column: `lo = min(column min, 0)`, `hi = max(column max, 0)`, `maxq = 2**bits - 1`,
`scale = (hi - lo) / maxq`, `zero = round(-lo / scale)`. A column of all zeros would give
`scale = 0`; use a scale of 1.0 there. A weight is stored as the integer
`q = clip(round(w / scale) + zero, 0, maxq)` and read back as `scale * (q - zero)`.
Use `np.round` (ties to even); the checks never place a value on a tie.

- `find_params(w, bits)`: the `(scale, zero)` of a block, each of shape `(d_out,)`.
- `quantize_to_grid(w, scale, zero, bits)`: store and read back, "fake-quantise": the
  values the quantised weights represent, as floats, for `w` of any shape broadcastable
  against `scale` and `zero`.

<details><summary>💡 Hint 1 — what to think about</summary>

What should the grid do for a column whose weights are all positive? Which integer is 0.0 then?
What happens to a weight outside the column's `[lo, hi]` range, and why can that happen at all
(think ahead: the weights this is called on later will not always be the ones the grid was fitted to)?
</details>

<details><summary>💡 Hint 2 — the approach in words</summary>

Take the column minima and maxima over the rows (axis 0), clamp them so they straddle zero,
divide the range by `maxq`, and round the zero point. To store a weight, divide by the scale, round,
add the zero point and clip to the integers the grid has; to read it back, undo the shift and scale.
Guard the division for the all-zero column before you divide by the scale.
</details>

In [ ]:
def find_params(w: np.ndarray, bits: int):
    """The scale and zero point of the asymmetric min-max grid for one group of weights.

    `w` has shape `(g, d_out)`; each output column gets its own grid. The range of a column is
    widened to include 0, `scale = (hi - lo) / (2**bits - 1)` (1.0 for an all-zero column) and
    `zero = np.round(-lo / scale)`. Returns `(scale, zero)`, each a float array of shape `(d_out,)`.

    Worked example: with bits 2 and the block [[-0.6, 1.0], [0.0, 2.0], [0.3, 4.0]], the first column
    runs from -0.6 to 0.3 and gives scale 0.3, zero 2; the second is all positive, so its range is
    widened down to 0 and it gives scale 4/3, zero 0.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement find_params")


def quantize_to_grid(w: np.ndarray, scale: np.ndarray, zero: np.ndarray, bits: int) -> np.ndarray:
    """Round `w` to the grid `(scale, zero)` and read it back as floats.

    `q = clip(np.round(w / scale) + zero, 0, 2**bits - 1)`, and the result is `scale * (q - zero)`.
    `w` may have any shape that broadcasts against `scale` and `zero` (a block of rows, or a single
    row of `d_out` weights). Returns a new float array of the shape of `w`.

    Worked example: bits 2, scale [0.3, 4/3], zero [2, 0], w = [0.5, 1.9] gives [0.3, 4/3]. The first
    weight lies above its column's grid (the top level is 0.3), so it is clipped to that level; 1.9 is
    nearest the level 4/3.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement quantize_to_grid")


def _check_grid() -> None:
    scale, zero = find_params(np.array([[-0.6, 1.0], [0.0, 2.0], [0.3, 4.0]]), 2)
    assert np.allclose(scale, [0.3, 4 / 3], rtol=0, atol=1e-12) and np.allclose(zero, [2, 0], rtol=0, atol=0), (
        f"find_params on the docstring block gave scale {np.round(scale, 4).tolist()}, zero {zero.tolist()}; "
        "expected scale [0.3, 1.3333] and zero [2, 0]: the range is (hi - lo) / (2**bits - 1) per COLUMN, and "
        "an all-positive column must be widened down to 0")
    assert scale.shape == (2,) and zero.shape == (2,), "scale and zero are one number per output column, shape (d_out,)"
    out = quantize_to_grid(np.array([0.5, 1.9]), np.array([0.3, 4 / 3]), np.array([2.0, 0.0]), 2)
    assert np.allclose(out, [0.3, 4 / 3], rtol=0, atol=1e-12), (
        f"quantize_to_grid gave {np.round(out, 4).tolist()}, expected [0.3, 1.3333]: a weight above the grid's "
        "top level must clip to it (clip the integer, not the float)")
    low = quantize_to_grid(np.array([-5.0, -5.0]), np.array([0.3, 4 / 3]), np.array([2.0, 0.0]), 2)
    assert np.allclose(low, [-0.6, 0.0], rtol=0, atol=1e-12), (
        f"quantize_to_grid gave {np.round(low, 4).tolist()} for weights far below the grid, expected [-0.6, 0.0]: "
        "they must clip to the LOWEST level (integer 0) as well")
    # a weight 0.49 of a step from a level belongs to that level and one 0.51 of a step away to the next (3 bits, zero point 3)
    levels = np.arange(8)
    for off in (-0.51, -0.49, 0.49, 0.51):
        probe = 0.25 * (levels - 3 + off)
        want = 0.25 * (np.clip(levels + (off > 0.5) - (off < -0.5), 0, 7) - 3)
        got = quantize_to_grid(probe, np.array([0.25]), np.array([3.0]), 3)
        assert np.allclose(got, want, rtol=0, atol=1e-12), (
            f"quantize_to_grid puts a weight {off:+.2f} of a step from a level on the wrong level: round to the NEAREST level, "
            "so anything closer than half a step belongs to the level it is nearest to (no bias up or down)")
    rng = np.random.default_rng(3)
    for bits in (2, 3, 4, 8):
        w = rng.normal(0, 1, (7, 5))
        s, z = find_params(w, bits)
        r = quantize_to_grid(w, s, z, bits)
        levels = max(len(np.unique(np.round(r[:, c] / s[c]))) for c in range(5))
        assert levels <= 2 ** bits, f"bits={bits}: a column used {levels} distinct levels, but the grid has {2 ** bits}"
        assert np.all(np.abs(r - w) <= s / 2 + 1e-12), (
            f"bits={bits}: a weight inside the range moved by more than half a step: round to the NEAREST level")
        zq = quantize_to_grid(np.zeros(5), s, z, bits)
        assert np.all(zq == 0.0), f"bits={bits}: 0.0 must be stored exactly (the zero point must be a whole number)"
    s0, z0 = find_params(np.zeros((4, 3)), 3)
    assert np.all(np.isfinite(s0)) and np.all(s0 > 0), "an all-zero column gives scale 0 and a division by zero: use scale 1.0 there"
    assert np.all(s0 == 1.0), "the scale of an all-zero column is 1.0"
    assert np.all(quantize_to_grid(np.zeros((4, 3)), s0, z0, 3) == 0.0), "an all-zero block must read back as zeros, not NaN"
    sp, zp = find_params(np.array([[1.0], [2.0], [3.0]]), 3)
    assert zp[0] == 0 and np.isclose(sp[0], 3 / 7), "an all-positive column's range starts at 0: zero point 0, scale hi / maxq"
    sn, zn = find_params(np.array([[-1.0], [-4.0]]), 3)
    assert zn[0] == 7 and np.isclose(sn[0], 4 / 7), "an all-negative column's range ends at 0: the highest level is 0.0 (zero point 7 at 3 bits)"
    sf, zf = find_params(np.array([[-0.2, -0.3], [0.9, 0.9]]), 3)
    assert np.all(zf == np.round(zf)), "a zero point must be a whole number (an integer level): round it"
    assert zf.tolist() == [1.0, 2.0], f"zero points {zf.tolist()}: round -lo / scale to the NEAREST integer (floor and ceil are both wrong here)"
    print("grid: the docstring examples, half-step accuracy, level counts, exact zeros and the all-zero column")


if __name__ == "__main__":
    _try("exercise 1", _check_grid)

## 3. Exercise 2: round-to-nearest, group by group

Round-to-nearest (RTN) is the baseline every method is judged against. It needs no data at all:
split the rows of `W` (the input channels) into consecutive **groups** of `group_size`, fit a grid
to each group with `find_params`, and round every weight to its group's grid. The last group is
shorter when `group_size` does not divide the number of rows; a `group_size` at least as large as
the number of rows makes one group for the whole matrix. Groups run **down the rows** (input
channels) and each output column has its own grid inside a group.

`rtn_quantize` returns the fake-quantised matrix: what the stored integers represent, as floats.
It must not change `W`.

<details><summary>💡 Hint 1 — what to think about</summary>

Which axis do the groups run along, and which axis does `find_params` take its minima over? What is
the shape of the block you hand it, and of the answer you put back? Do two groups share anything?
</details>

<details><summary>💡 Hint 2 — the approach in words</summary>

Make an output array, walk the starting rows `0, g, 2g, ...`, slice the block of up to `g` rows, fit the
grid with your own `find_params`, round the block with your own `quantize_to_grid` and write it into the
output. Python's slicing already handles the short last block.
</details>

In [ ]:
def rtn_quantize(W: np.ndarray, bits: int, group_size: int) -> np.ndarray:
    """Round-to-nearest quantisation of `W` (shape `(d_in, d_out)`) with groups of `group_size` rows.

    Each group of consecutive rows, and each output column inside it, gets its own grid from
    `find_params`; `group_size` is at least 1. The last group may be shorter. Returns a new float array of the
    shape of `W` holding the values the stored integers represent; `W` is not changed.

    Worked example: W has 6 rows and group_size is 4: rows 0-3 share one grid per column and rows 4-5
    share another. With group_size 6 or more, all six rows share one.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement rtn_quantize")


def _check_rtn() -> None:
    rng = np.random.default_rng(11)
    W = rng.normal(0, 1, (10, 3))
    W_in = W.copy()
    got = rtn_quantize(W, 3, 4)
    assert np.array_equal(W, W_in), "rtn_quantize changed its input: build a new array"
    assert got.shape == W.shape and got.dtype == np.float64, "the result must have the shape of W and be float64"
    for lo, hi in ((0, 4), (4, 8), (8, 10)):
        s, z = find_params(W[lo:hi], 3)
        assert np.allclose(got[lo:hi], quantize_to_grid(W[lo:hi], s, z, 3), rtol=0, atol=1e-12), (
            f"rows {lo}-{hi - 1} are not rounded on their own group's grid: groups run down the ROWS, the last "
            "group is shorter when group_size does not divide the row count, and each group is fitted separately")
    W2 = W.copy()
    W2[0, 0] = 40.0                               # an outlier in group 1 must not touch group 2
    got2 = rtn_quantize(W2, 3, 4)
    assert np.array_equal(got2[4:], got[4:]), "a change inside one group altered another group: groups must not share a grid"
    one = rtn_quantize(W, 3, 10)
    s, z = find_params(W, 3)
    assert np.allclose(one, quantize_to_grid(W, s, z, 3), rtol=0, atol=1e-12), "group_size >= rows means ONE group for the whole matrix"
    assert np.array_equal(rtn_quantize(W, 3, 99), one), "a group_size larger than the matrix is one group, not an error"
    ints = rtn_quantize(np.round(W * 10).astype(np.int64), 3, 4)
    assert ints.dtype == np.float64 and np.allclose(ints, rtn_quantize(np.round(W * 10), 3, 4), rtol=0, atol=1e-12), (
        "an integer matrix must give the same float result as the same numbers as floats: allocate the output as float64")
    print("rtn_quantize: group boundaries, the short last group, independent groups, the input left alone")


if __name__ == "__main__":
    _try("exercise 2", _check_rtn, needs=("exercise 1",))

### What round-to-nearest costs

Now that `rtn_quantize` works, here is the baseline: the relative output error of each of the six
matrices on the held-out text, at 4 and 3 bits, and how the group size trades accuracy for storage.
Every number is computed by the cell.

In [ ]:
GROUP_SIZES = (4, 8, 16, 48)       # all divide both 48 and 192, so no group is short


def _show_rtn_baseline() -> None:
    rows = []
    for name in QUANTISED:
        W, X = PARAMS[name], HELD[name]
        rows.append([f"{name} {W.shape[0]}x{W.shape[1]}", *[_percent(rel_output_error(X, W, rtn_quantize(W, b, GROUP_SIZE)))
                                                              for b in BITS]])
    print(f"relative output error of RTN on held-out text, groups of {GROUP_SIZE} input channels")
    _print_table(["matrix", *[f"{b}-bit" for b in BITS]], rows)
    print()
    rows = []
    for g in GROUP_SIZES:
        errs = {b: np.mean([rel_output_error(HELD[n], PARAMS[n], rtn_quantize(PARAMS[n], b, g)) for n in QUANTISED])
                for b in BITS}
        rows.append([str(g), *[f"{bits_per_weight(b, g):.2f}" for b in BITS], *[_percent(errs[b]) for b in BITS]])
    print("group size against storage and mean error over the six matrices")
    _print_table(["group", *[f"bits/weight at {b}" for b in BITS], *[f"{b}-bit error" for b in BITS]], rows)
    smaller = [bits_per_weight(4, g) for g in GROUP_SIZES]
    print(f"\nstorage counts one 16-bit scale and one zero point per group, so a group of {GROUP_SIZES[0]} costs "
          f"{smaller[0]:.2f} bits per weight at 4 bits and a group of {GROUP_SIZES[-1]} costs {smaller[-1]:.2f}")


if __name__ == "__main__":
    _try("RTN baseline", _show_rtn_baseline, needs=("exercise 2",))

## 4. Exercise 3: the output error is a quadratic form

RTN rounds each weight to its nearest grid point. That minimises the error in the **weights**, and
never looks at the data. What the model feels is the error in the layer's **output**. For calibration
inputs `X` (one row per token, `n` rows) and a weight error `ΔW = W - W_hat`:

`output error = (1/n) Σ_t ‖x_t ΔW‖²  =  (1/n) ‖X ΔW‖²_F  =  trace(ΔWᵀ H ΔW)`, with `H = XᵀX / n`.

`H` is, up to a constant factor, the **Hessian** of the layer-wise objective: the objective is quadratic in the
weights, so its second derivative is a constant matrix that depends only on the inputs. (The GPTQ paper writes
`2 X Xᵀ` with the tokens as columns; a positive constant factor changes nothing below.) Two facts follow.
Every output column of `W` has the *same* `H`. And a weight error costs different amounts in different
directions: along an eigenvector of `H` with a large eigenvalue it is expensive, along one with a small
eigenvalue it is almost free. Plain rounding does not know which is which; GPTQ will.

- `layer_hessian(X)`: `XᵀX / n`, shape `(d_in, d_in)`.
- `quad_form_error(H, dW)`: `trace(dWᵀ H dW)` for `dW` of shape `(d_in, d_out)`, as a float.

<details><summary>💡 Hint 1 — what to think about</summary>

Check the shapes. `X` is `(n, d_in)` and `H` must be `(d_in, d_in)`: which of `XᵀX` and `XXᵀ` is that?
Is the division by `n` part of the definition or not, and what would happen to the identity above without it?
</details>

<details><summary>💡 Hint 2 — the approach in words</summary>

`layer_hessian` is one matrix product and one division. For `quad_form_error`, multiply `dW` into `H`
first, then take the elementwise product with `dW` and sum everything: that is the trace of
`dWᵀ (H dW)` without forming a matrix you then throw away.
</details>

In [ ]:
def layer_hessian(X: np.ndarray) -> np.ndarray:
    """`H = XᵀX / n` for calibration inputs `X` of shape `(n, d_in)`: a `(d_in, d_in)` float array.

    Worked example: X = [[1, 0], [0, 2], [1, 2]] gives XᵀX = [[2, 2], [2, 8]], and with n = 3,
    H = [[2/3, 2/3], [2/3, 8/3]].
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement layer_hessian")


def quad_form_error(H: np.ndarray, dW: np.ndarray) -> float:
    """`trace(dWᵀ H dW)` as a Python float, for `H` of shape `(d_in, d_in)` and `dW` of shape `(d_in, d_out)`.

    This is the layer's output error written with `H` alone. Worked example: with the `H` above and
    dW = [[1], [0]] (one output channel, an error of 1 on the first input channel only) the answer is
    H[0, 0] = 2/3; with dW = [[1, 0], [0, 1]] it is trace(H) = 10/3.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement quad_form_error")


def _check_hessian() -> None:
    X = np.array([[1.0, 0.0], [0.0, 2.0], [1.0, 2.0]])
    H = layer_hessian(X)
    assert H.shape == (2, 2) and np.allclose(H, [[2 / 3, 2 / 3], [2 / 3, 8 / 3]], rtol=0, atol=1e-12), (
        f"layer_hessian gave {np.round(H, 4).tolist()}, expected [[0.6667, 0.6667], [0.6667, 2.6667]]: "
        "H = XᵀX / n with the tokens as ROWS of X (so d_in x d_in), divided by the number of tokens")
    assert np.allclose(quad_form_error(H, np.array([[1.0], [0.0]])), 2 / 3, rtol=0, atol=1e-12) and \
        np.allclose(quad_form_error(H, np.eye(2)), 10 / 3, rtol=0, atol=1e-12), (
            "quad_form_error is trace(dWᵀ H dW): an error on one input channel costs that channel's H diagonal entry")
    rng = np.random.default_rng(5)
    Xr = rng.normal(0, 1, (40, 6)) @ rng.normal(0, 1, (6, 6))
    W, W_hat = rng.normal(0, 1, (6, 4)), rng.normal(0, 1, (6, 4))
    direct, via_h = output_error(Xr, W, W_hat), quad_form_error(layer_hessian(Xr), W - W_hat)
    assert type(via_h) is float, "quad_form_error must return a plain Python float"
    assert abs(direct - via_h) <= 1e-9 * abs(direct), (
        f"the identity fails: computed through X the output error is {direct:.6f}, through H it is {via_h:.6f}. "
        "Either H is not XᵀX / n, or the trace is wrong (a sum of dW * (H @ dW), not dW @ H @ dW)")
    print("hessian: H = XᵀX / n, and the quadratic form equals the output error computed through X")


if __name__ == "__main__":
    _try("exercise 3", _check_hessian)

### The direction of an error matters

The next cell takes the input `Wf1` sees, builds its Hessian from a calibration set, and compares
three weight errors of exactly the same size (Frobenius norm 1): one pointing along a random
direction of the input space, one along the eigenvector of `H` with the largest eigenvalue, and
one along the smallest. The two cost columns, via `H` and via `X`, are your identity again on real activations: they
should agree.

In [ ]:
def _show_direction() -> None:
    calib = calibration_set(POOL, CALIB_TOKENS, 0)
    X = calib["Wf1"]
    H = layer_hessian(X)
    values, vectors = np.linalg.eigh(H)
    rng = np.random.default_rng(SYNTH_SEED)
    random_dir = rng.normal(0, 1, H.shape[0])
    random_dir /= np.linalg.norm(random_dir)
    out = rng.normal(0, 1, PARAMS["Wf1"].shape[1])
    out /= np.linalg.norm(out)
    rows = []
    for label, u in (("random direction", random_dir), ("largest eigenvector", vectors[:, -1]),
                     ("smallest eigenvector", vectors[:, 0])):
        dW = np.outer(u, out)
        rows.append([label, f"{np.linalg.norm(dW):.3f}", f"{quad_form_error(H, dW):.5f}",
                     f"{output_error(X, np.zeros_like(dW), -dW):.5f}"])
    _print_table(["weight error pointing along", "‖ΔW‖", "cost via H", "cost via X"], rows)
    print(f"\nthe eigenvalues of H run from {values[0]:.5f} to {values[-1]:.5f}, a ratio of {values[-1] / values[0]:.0f}: "
          "the same amount of weight error costs that much more in the stiffest direction than in the softest")


if __name__ == "__main__":
    _try("hessian directions", _show_direction, needs=("exercise 3",))

## 5. Exercise 4: rounding one weight, and compensating with the rest

Take one output column `w` (length `d_in`) and its Hessian `H`. Suppose weight `i` must be set to a
grid value `q`. The change `δ = w' - w` costs `δᵀ H δ`. Weight `i` is forced: `δ_i = c`, with
`c = q - w_i`. The other weights are free to move, and the **Optimal Brain Surgeon** question (Hassibi and Stork's
framework; see `claims.yaml`) is where they should go. Minimise `δᵀ H δ` subject to `eᵢᵀ δ = c` with a Lagrange multiplier `λ`:

- the gradient condition `H δ = λ eᵢ` gives `δ = λ H⁻¹ eᵢ`;
- the constraint `δ_i = c` gives `λ (H⁻¹)ᵢᵢ = c`, so `λ = c / (H⁻¹)ᵢᵢ`;
- therefore `δ = c · (H⁻¹)[:, i] / (H⁻¹)ᵢᵢ`, and the cost of the best choice is `c² / (H⁻¹)ᵢᵢ`.

That is equation (2) of the GPTQ paper. Compare it with *not* compensating, which means `δ = c · eᵢ` and
costs `c² Hᵢᵢ`. The ratio of the two, `Hᵢᵢ (H⁻¹)ᵢᵢ`, is never below 1, and is exactly 1 when input channel `i` has a
zero mean product with every other channel (`Hᵢⱼ = 0` for all `j ≠ i`; for inputs with zero mean, that is the same as being
uncorrelated with them). All the output columns of a matrix share the same `H`, so one function can update a whole matrix at once.

`obs_update(Hinv, w, i, q)` takes the **inverse** Hessian over the entries of `w`, `w` of shape `(d,)`
or `(d, k)` (k output columns), the index `i` and the forced value `q` (a number, or `k` numbers, one per
column), and returns the updated weights as a new array. Entry `i` of every column is then exactly `q`:
assign it, do not rely on the arithmetic landing on it.

<details><summary>💡 Hint 1 — what to think about</summary>

Which column of the inverse Hessian does the update point along, and what scales it? What does the
update do to entry `i` itself if `c` is the whole forced change? With `k` columns, what is `c` for
each one, and how do you build an update for all of them at once without a Python loop?
</details>

<details><summary>💡 Hint 2 — the approach in words</summary>

Form `c = q - w[i]` (one number per column), divide by `Hinv[i, i]`, and add the outer product of
`Hinv[:, i]` with that. Then overwrite entry `i` with `q`. Work on a copy of `w`.
</details>

In [ ]:
def obs_update(Hinv: np.ndarray, w: np.ndarray, i: int, q) -> np.ndarray:
    """The Optimal Brain Surgeon update: set weight `i` to `q` and move every other weight by the
    amount that adds the least output error.

    `Hinv` is the inverse Hessian `(d, d)`; `w` is `(d,)` or `(d, k)`; `q` is a number or `k` numbers.
    Returns a new array shaped like `w` in which entry `i` is exactly `q` and the others have moved by
    `(q - w[i]) / Hinv[i, i] * Hinv[:, i]`. `w` is not changed.

    Worked example: Hinv [[2, 1], [1, 1]], w [1, 1], i 0, q 0 -> [0, 0.5]. Not compensating would leave
    the second weight at 1; the optimum moves it half way towards 0.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement obs_update")


def _check_obs() -> None:
    Hinv = np.array([[2.0, 1.0], [1.0, 1.0]])
    got = obs_update(Hinv, np.array([1.0, 1.0]), 0, 0.0)
    assert np.allclose(got, [0.0, 0.5], rtol=0, atol=1e-12), (
        f"obs_update on the docstring example gave {got.tolist()}, expected [0.0, 0.5]: the others move by "
        "(q - w[i]) / Hinv[i, i] times column i of Hinv, with the sign that carries w[i] to q")
    rng = np.random.default_rng(21)
    A = rng.normal(0, 1, (6, 6))
    H = A @ A.T + 0.5 * np.eye(6)
    Hinv = np.linalg.inv(H)
    w = rng.normal(0, 1, 6)
    w_before = w.copy()
    for i in (0, 3, 5):
        got = obs_update(Hinv, w, i, 0.7)
        assert got[i] == 0.7, f"entry {i} must be exactly q after the update; assign it rather than trusting the arithmetic"
        rest = [j for j in range(6) if j != i]
        # the brute-force optimum: minimise δᵀ H δ over the free entries, with δ_i fixed at q - w_i
        delta_free = np.linalg.solve(H[np.ix_(rest, rest)], -H[rest, i] * (0.7 - w[i]))
        assert np.allclose(got[rest], w[rest] + delta_free, rtol=0, atol=1e-9), (
            f"i={i}: the other weights are not at the constrained optimum. The update is a multiple of column i of "
            "the INVERSE Hessian, divided by Hinv[i, i]; using H itself, or forgetting the division, gives the wrong place")
    assert np.array_equal(w, w_before), "obs_update changed its input: work on a copy"
    W = rng.normal(0, 1, (6, 3))
    q = np.array([0.1, -0.4, 0.9])
    many = obs_update(Hinv, W, 2, q)
    for col in range(3):
        assert np.allclose(many[:, col], obs_update(Hinv, W[:, col], 2, q[col]), rtol=0, atol=1e-12), (
            "a matrix of k columns must be updated column by column with its own q, all with the same Hinv")
    assert np.all(many[2] == q), "row i of every column must equal that column's q"
    print("obs_update: the docstring example, the constrained optimum from a direct solve, several columns")


if __name__ == "__main__":
    _try("exercise 4", _check_obs)

### How much can compensation buy?

The ratio `Hᵢᵢ (H⁻¹)ᵢᵢ` says how much cheaper the best compensated rounding of channel `i` is than an
uncompensated one, whatever the size of the rounding error. The next cell computes it for every input
channel of each matrix, from the same calibration set and the same dampening GPTQ will use.

In [ ]:
def _show_compensation_ratio() -> None:
    calib = calibration_set(POOL, CALIB_TOKENS, 0)
    rows = []
    for name in QUANTISED:
        H = layer_hessian(calib[name])
        Hd = H + PERCDAMP * np.mean(np.diag(H)) * np.eye(len(H))
        ratio = np.diag(Hd) * np.diag(np.linalg.inv(Hd))
        rows.append([f"{name} ({len(H)} channels)", f"{ratio.min():.2f}", f"{np.median(ratio):.2f}", f"{ratio.max():.2f}"])
    print("how many times cheaper the best compensated rounding of one weight is than an uncompensated one")
    _print_table(["matrix", "smallest", "median", "largest"], rows)
    print("\na ratio of 1 means the channel has a zero mean product with every other channel, so there is nothing to compensate with")


if __name__ == "__main__":
    _try("compensation ratio", _show_compensation_ratio, needs=("exercise 3",))

## 6. Exercise 5: GPTQ

Now apply that update to every weight in turn. Walk the input channels (the rows of `W`) in a fixed order
`i = 0, 1, 2, ...`. At step `i`, round row `i` to its grid, then compensate all rows **below** it with the update
of the last exercise, for every output column at once, because they share `H`. Each step needs the inverse
Hessian of the channels not yet rounded; dropping channel `i` from an inverse is one rank-one update (equation 3
of the paper), and the lesson shows below that this gives GPTQ's rounding decisions exactly.

The paper's practical trick: write `H⁻¹ = Uᵀ U` with `U` upper triangular (a Cholesky factor). Let `R` be the
inverse Hessian of the channels not yet rounded when step `i` starts, with channel `i` first. Then
`U[i, i]² = R[0, 0]` and `U[i, j] = R[0, j - i] / U[i, i]` for `j > i`: the factor holds, row by row, what
`obs_update` needs from every successive inverse, so one factorisation replaces all the rank-one downdates,
and it is numerically steadier. Your job is to rewrite the update of exercise 4, for the rows below `i`, in
terms of `U[i, i]` and the part of row `i` of `U` to the right of the diagonal.

Three details decide whether yours matches the lesson's. The first two follow the authors' reference code (see
`claims.yaml`); the third is the lesson's own rule, and the next cell says where the reference code differs:

- **Dampening.** Before inverting, add `λ = percdamp × mean(diag H)` to the diagonal; a Hessian from fewer
  tokens than channels has no inverse without it.
- **Dead channels.** A channel that is exactly zero on every calibration token has `H[i, i] = 0`. Set that entry
  to 1 *before* taking the mean for `λ`, and set that channel's weight row to zero.
- **Group grids.** When row `i` starts a group, fit the group's grid to the group's rows *as they are now*,
  after the compensation of every earlier row, not to the original weights.

Write `gptq_quantize` through the factor: take `U` and the dead mask from `damped_inverse_factor` and write the
update in terms of `U`, without calling `obs_update` or the explicit version given in the next cell. The checks
swap in a factor routine of their own to make sure of it.

- `damped_inverse_factor(H, percdamp)`: returns `(U, dead)`, the upper factor and the boolean dead-channel mask.
- `gptq_quantize(W, H, bits, group_size, percdamp)`: returns the fake-quantised matrix.

<details><summary>💡 Hint 1 — what to think about</summary>

`np.linalg.cholesky` returns a lower triangle `L` with `A = L Lᵀ`. Which of `L` and `Lᵀ` has the entries you
need to the right of the diagonal? Write exercise 4's update with `R` in place of `Hinv` for the rows below
`i`, then replace `R[0, 0]` and `R[0, j - i]` using the two identities. Which rows does step `i` change, and
which row is final once it has run? On which array does `find_params` look, and when?
</details>

<details><summary>💡 Hint 2 — the approach in words</summary>

Factor: copy `H`, find the exact-zero diagonal entries, set them to 1, add `percdamp` times the mean diagonal
to the diagonal, invert, take the Cholesky factor and transpose it. Loop: copy `W`, zero the dead rows, then
for each row fit the grid if the row starts a group, round the row and store it in the output. To move the rows
below it, write exercise 4's update with `R` in place of `Hinv`, substitute the two identities, and check which
rows step `i` changes.
</details>

In [ ]:
def damped_inverse_factor(H: np.ndarray, percdamp: float):
    """The upper-triangular Cholesky factor `U` of the damped inverse Hessian, with `Uᵀ U = (H' + λ I)⁻¹`.

    `H'` is `H` with every exactly-zero diagonal entry set to 1 (these channels are returned in the
    boolean mask `dead`); `λ = percdamp * mean(diag(H'))`. `H` is not changed. Returns `(U, dead)`.

    Worked example: H = diag(4, 1, 0) and percdamp 0.5: dead is [False, False, True]; the fixed diagonal
    is [4, 1, 1] with mean 2, so λ = 1 and the damped matrix is diag(5, 2, 2); U is diag(1/√5, 1/√2, 1/√2).
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement damped_inverse_factor")


def gptq_quantize(W: np.ndarray, H: np.ndarray, bits: int, group_size: int, percdamp: float = PERCDAMP) -> np.ndarray:
    """GPTQ for one matrix: round the rows of `W` (shape `(d_in, d_out)`) in order, compensating every
    later row for each rounding error through the Hessian `H` of shape `(d_in, d_in)`.

    Takes `U` and the dead mask from `damped_inverse_factor` (and builds no inverse Hessian of its own),
    zeroes the weight rows of dead channels, and for each row `i`: fits a grid with `find_params` on the
    group's current rows when `i` starts a group, rounds row `i` with `quantize_to_grid`, and moves the rows
    below it by the Optimal Brain Surgeon update of exercise 4, written through `U`: it does not call
    `obs_update` or `gptq_explicit`. Returns a new array of the shape of `W` holding the values the stored
    integers represent; `W` and `H` are not changed.

    Worked example: if `H` is diagonal nothing is compensated and the result equals
    `rtn_quantize(W, bits, group_size)`; a dead channel's row comes back as zeros.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement gptq_quantize")

### The check: GPTQ against the slow way

The function below is GPTQ written the way the derivation reads: after rounding row `i`, apply **your** `obs_update`
to all the rows still to be rounded, then drop channel `i` from the inverse Hessian with one rank-one downdate. It
takes the damped inverse Hessian as an argument, assumes no channel is dead, needs no Cholesky factor and costs more.
It is a yardstick, not a shortcut: the check runs your `gptq_quantize` with this function and `obs_update` switched
off, and with a `damped_inverse_factor` that doubles whatever dampening it is asked for, so only a version that really
works through the factor it is handed gives the right answer. Its two optional arguments, `fit_to` and `stats`, are for a
measurement further down.

The check compares the two on small matrices with several groups, entry by entry to within 1e-9. It is where your
reading of "fit the group's grid to the current rows" is tested. That rule is this lesson's own. When the authors'
reference code is asked for groups, it applies the rule whenever a group starts a block of 128 columns (as every group
does when the group size is 128, the size the AWQ paper uses throughout; see `claims.yaml`), but it updates the matrix in
blocks, so for a group that starts inside a block (as most groups of 16 do) it fits the grid to weights compensated only
up to the start of that block. Its default is no groups at all: one grid per output channel, fitted once to the original
weights before the loop. The lesson always uses the fully compensated rows.

In [ ]:
@contextlib.contextmanager
def _swapped(**replacements):
    """Replace some functions of this notebook for the length of a `with` block, so that a check can see which
    pieces a function really uses."""
    saved = {name: globals()[name] for name in replacements}
    globals().update(replacements)
    try:
        yield
    finally:
        globals().update(saved)


def _refuse(name: str) -> Callable:
    def refuse(*args, **kwargs):
        raise AssertionError(f"gptq_quantize called {name}(): exercise 5 asks for the update written through U, the factor "
                             "damped_inverse_factor returns, not for a call to exercise 4's obs_update or to the given explicit version")
    return refuse


def gptq_explicit(W: np.ndarray, Hinv: np.ndarray, bits: int, group_size: int, fit_to: str = "current",
                  stats: dict | None = None) -> np.ndarray:
    """GPTQ by repeated Optimal Brain Surgeon updates and explicit downdates of the damped inverse Hessian `Hinv`. Given.

    Assumes no dead channel (the caller zeroes the weight rows of dead channels) and uses YOUR obs_update. `fit_to` says
    what a group's grid is fitted to when the loop reaches the group's first row: "current", the group's rows as they are
    by then (the rule of exercise 5), or "original", the weights as they were before the loop. If `stats` is a dict,
    `stats["clipped"]` counts the rounded values that fell outside their grid and were clipped to its end."""
    if fit_to not in ("current", "original"):
        raise ValueError("fit_to is 'current' or 'original'")
    W, Hinv = np.array(W, dtype=np.float64), np.array(Hinv, dtype=np.float64)
    original = W.copy()
    Q = np.zeros_like(W)
    for i in range(W.shape[0]):
        # the rows i.. are the ones still to be rounded; Hinv is their inverse Hessian, indexed from 0
        if i % group_size == 0:
            scale, zero = find_params((W if fit_to == "current" else original)[i:i + group_size], bits)
        if stats is not None:
            level = np.round(W[i] / scale) + zero
            stats["clipped"] = stats.get("clipped", 0) + int(np.count_nonzero((level < 0) | (level > 2 ** bits - 1)))
        Q[i] = quantize_to_grid(W[i], scale, zero, bits)
        W[i:] = obs_update(Hinv, W[i:], 0, Q[i])
        Hinv = Hinv[1:, 1:] - np.outer(Hinv[1:, 0], Hinv[0, 1:]) / Hinv[0, 0]
    return Q


def _check_gptq() -> None:
    U, dead = damped_inverse_factor(np.diag([4.0, 1.0, 0.0]), 0.5)
    assert np.asarray(dead).tolist() == [False, False, True], f"dead mask {np.asarray(dead).tolist()}: a channel is dead when its diagonal entry is exactly 0"
    assert np.allclose(U, np.diag([5 ** -0.5, 2 ** -0.5, 2 ** -0.5]), rtol=0, atol=1e-12), (
        f"U on the docstring example is {np.round(np.diag(U), 4).tolist()} on its diagonal, expected [0.4472, 0.7071, 0.7071]: "
        "set the dead entry to 1 BEFORE taking the mean for λ, then add λ to the whole diagonal")
    rng = np.random.default_rng(31)
    A = rng.normal(0, 1, (5, 5))
    H = A @ A.T / 5
    U, dead = damped_inverse_factor(H, 0.01)
    assert np.allclose(U, np.triu(U), rtol=0, atol=0), "U must be UPPER triangular: transpose the Cholesky factor numpy gives you"
    lam = 0.01 * np.mean(np.diag(H))
    assert np.allclose(U.T @ U, np.linalg.inv(H + lam * np.eye(5)), rtol=0, atol=1e-9), (
        "Uᵀ U should be the inverse of the damped Hessian, with λ = percdamp * the mean of the diagonal")
    Xd = rng.normal(0, 1, (40, 5))
    Xd[:, 1] = 0.0                             # a dead channel
    Xd[:, 3] *= 1e-5                           # a tiny but live one: its diagonal entry is about 1e-10, not 0
    H_dead = Xd.T @ Xd / 40
    H_before = H_dead.copy()
    _, dead_mask = damped_inverse_factor(H_dead, 0.01)
    assert np.array_equal(H_dead, H_before), "damped_inverse_factor changed the caller's H (setting the dead entry to 1): work on a copy"
    assert np.asarray(dead_mask).tolist() == [False, True, False, False, False], (
        f"dead mask {np.asarray(dead_mask).tolist()}: only a channel whose diagonal entry is EXACTLY 0 is dead, a tiny one is live")
    H_tiny = H_dead.copy()
    H_tiny[3, :] = 0.0
    H_tiny[:, 3] = 0.0
    H_tiny[3, 3] = 1e-200                      # live, positive, and far below any tolerance you might pick
    assert np.asarray(damped_inverse_factor(H_tiny, 0.01)[1]).tolist() == [False, True, False, False, False], (
        "a diagonal entry of 1e-200 is not zero: a channel is dead only when its entry is EXACTLY 0, so compare with == 0, "
        "not with a tolerance or a threshold")
    # a diagonal Hessian means no input channel carries information about another: nothing to compensate
    W = rng.normal(0, 1, (8, 3))
    diag_h = np.diag(rng.uniform(0.5, 2.0, 8))
    plain = rtn_quantize(W, 3, 4)
    got = gptq_quantize(W, diag_h, 3, 4)
    assert np.allclose(got, plain, rtol=0, atol=1e-12), (
        "with a diagonal Hessian GPTQ must reproduce round-to-nearest exactly: a step may only change the rows BELOW "
        "it, by an amount that comes from row i of U to the right of the diagonal, which is all zeros here. Check the "
        "triangle you read and the rows you update")
    # correlated inputs: the compensation must pay for itself
    X = rng.normal(0, 1, (400, 8)) @ rng.normal(0, 1, (8, 8))
    Hc = X.T @ X / 400
    Wc = rng.normal(0, 1, (8, 5))
    W_in, H_in = Wc.copy(), Hc.copy()
    g, r = gptq_quantize(Wc, Hc, 3, 4), rtn_quantize(Wc, 3, 4)
    assert np.array_equal(Wc, W_in) and np.array_equal(Hc, H_in), "gptq_quantize changed W or H: work on copies"
    assert g.shape == Wc.shape, "the result must have the shape of W"
    e_g, e_r = output_error(X, Wc, g), output_error(X, Wc, r)
    assert e_g < 0.8 * e_r, (
        f"on correlated inputs GPTQ's output error ({e_g:.4f}) should be clearly below round-to-nearest's ({e_r:.4f}). "
        "Either no error reaches the later rows (the factor is lower triangular, or the update is dropped), or it is divided by the wrong entry")
    assert weight_error(Wc, g) > weight_error(Wc, r), "GPTQ should round some weights away from their nearest level, so its weight error is larger"

    # The same loop against explicit elimination, with several groups so that the grids are fitted to compensated rows. The
    # factor routine is swapped for one that doubles the dampening it is asked for, and the two shortcuts are switched off:
    # an answer that does not work through the factor it is handed (or ignores percdamp) cannot match.
    real_factor = damped_inverse_factor

    def doubled_factor(H_, percdamp):
        return real_factor(H_, 2 * percdamp)

    def against_explicit(W_, H_, bits_, group_, percdamp=None):
        pd = PERCDAMP if percdamp is None else percdamp
        U2, dead2 = real_factor(H_, 2 * pd)
        dead2 = np.asarray(dead2, dtype=bool)
        W_zeroed = np.array(W_, dtype=np.float64)
        W_zeroed[dead2] = 0.0
        want = gptq_explicit(W_zeroed, U2.T @ U2, bits_, group_)            # uses your obs_update
        with _swapped(damped_inverse_factor=doubled_factor, gptq_explicit=_refuse("gptq_explicit"), obs_update=_refuse("obs_update")):
            got = gptq_quantize(W_, H_, bits_, group_) if percdamp is None else gptq_quantize(W_, H_, bits_, group_, percdamp)
        return got, want, ~dead2

    d, group, bits = 12, 4, 2
    Xm = rng.normal(0, 1, (300, d)) @ (np.eye(d) + 1.5 * rng.normal(0, 1, (d, d)))
    Hm, Wm = Xm.T @ Xm / 300, rng.normal(0, 1, (d, 5))
    for percdamp in (None, 0.1):
        got, want, _ = against_explicit(Wm, Hm, bits, group, percdamp)
        gap = float(np.max(np.abs(got - want)))
        assert gap < 1e-9, (
            f"on {d} channels in {d // group} groups of {group} (2 bits) gptq_quantize differs from explicit elimination by {gap:.1e}. "
            "Look at, in this order: which array each group's grid is fitted to (the group's rows as they are when its first row is "
            "reached, after the compensation of the rows before it, not the original weights); the triangle of U you read and the "
            "division by U[i, i]; whether the U comes from damped_inverse_factor and percdamp reaches it. This check also runs your "
            "obs_update, so if exercise 4 does not pass, fix that first")
    # dead channels: the live rows must agree with explicit elimination too, which needs the dead rows zeroed before any grid is fitted
    dead_h = Hc.copy()
    dead_h[3, :] = 0.0
    dead_h[:, 3] = 0.0
    W_dead = Wc.copy()
    W_dead[3] = [4.0, -4.0, 4.0, -4.0, 4.0]    # large weights on the dead channel: left in its group, they would widen the group's grid
    got, want, live = against_explicit(W_dead, dead_h, 3, 4)
    assert np.all(got[3] == 0.0), "a dead input channel's weight row must come back as exact zeros"
    assert np.all(np.isfinite(got)), "a dead channel produced NaN or inf: set its Hessian entry to 1 before inverting"
    gap = float(np.max(np.abs(got[live] - want[live])))
    assert gap < 1e-9, (
        f"with a dead channel the live rows differ from explicit elimination by {gap:.1e}: zero the dead channel's weight row "
        "BEFORE the loop, so that it cannot widen the grid of the group it sits in, and take the dead mask from damped_inverse_factor")
    print("gptq: the factor, GPTQ = RTN for a diagonal Hessian, lower output error on correlated inputs, equal to explicit "
          "elimination over several groups, dead channels")


if __name__ == "__main__":
    _try("exercise 5", _check_gptq, needs=("exercise 1", "exercise 2", "exercise 4"))

### GPTQ against the slow way, on the real matrices

The same comparison on matrices of the trained model, with their own Hessians and the lesson's groups, and with a
much heavier dampening as well, to see that `percdamp` is carried through. These matrices have no dead channel, so
the slow way can build its damped inverse Hessian directly.

In [ ]:
def _check_explicit() -> None:
    calib = calibration_set(POOL, CALIB_TOKENS, 0)
    for name, damp in (("Wv", PERCDAMP), ("Wf2", PERCDAMP), ("Wf2", 20 * PERCDAMP)):
        W, H = PARAMS[name], layer_hessian(calib[name])
        assert np.all(np.diag(H) > 0), f"{name} has a dead input channel, which this comparison does not handle"
        Hinv = np.linalg.inv(H + damp * np.mean(np.diag(H)) * np.eye(len(H)))
        fast, slow = gptq_quantize(W, H, 3, GROUP_SIZE, damp), gptq_explicit(W, Hinv, 3, GROUP_SIZE)
        worst = float(np.max(np.abs(fast - slow)))
        assert worst < 1e-9, (
            f"{name} with percdamp {damp}: the Cholesky version and the explicit-update version disagree (the largest "
            f"difference is {worst:.2e}). If exercise 4 passed, look at how gptq_quantize fits each group's grid (to the "
            "CURRENT rows, not the original weights), at which triangle of the factor it reads, at the division by "
            "U[i, i], and at whether percdamp reaches damped_inverse_factor")
        print(f"  {name} {W.shape[0]}x{W.shape[1]}, percdamp {damp:g}: Cholesky GPTQ and explicit-update GPTQ agree to within 1e-9")


if __name__ == "__main__":
    _try("GPTQ equals explicit OBS", _check_explicit, needs=("exercise 3", "exercise 4", "exercise 5"))

### What GPTQ does to the weights, and to the output

The next cell quantises all six matrices to 3 bits with RTN and with GPTQ, from the same calibration set, and
prints two errors for each: the error in the weights themselves (which RTN minimises), and the error in the
output on held-out text (which GPTQ minimises, on the calibration text).

In [ ]:
def _show_gptq_vs_rtn() -> None:
    calib = calibration_set(POOL, CALIB_TOKENS, 0)
    bits = 3
    rows, tot = [], np.zeros(4)
    for name in QUANTISED:
        W, H = PARAMS[name], layer_hessian(calib[name])
        r, g = rtn_quantize(W, bits, GROUP_SIZE), gptq_quantize(W, H, bits, GROUP_SIZE)
        vals = [weight_error(W, r), weight_error(W, g), rel_output_error(HELD[name], W, r), rel_output_error(HELD[name], W, g)]
        tot += vals
        rows.append([name, f"{vals[0]:.3f}", f"{vals[1]:.3f}", _percent(vals[2]), _percent(vals[3]),
                     f"{100 * np.mean(g != r):.0f}%"])
    print(f"{bits}-bit quantisation of each matrix, calibrated on {CALIB_TOKENS} tokens, output error on held-out text")
    _print_table(["matrix", "weight err RTN", "weight err GPTQ", "output err RTN", "output err GPTQ", "stored value differs"], rows)
    print(f"\nsummed over the six matrices GPTQ's weight error is {tot[1] / tot[0]:.2f} times RTN's, and its output "
          f"error is {tot[3] / tot[2]:.2f} times RTN's")


if __name__ == "__main__":
    _try("GPTQ against RTN", _show_gptq_vs_rtn, needs=("exercise 2", "exercise 3", "exercise 5"))

### Which weights should a group's grid be fitted to?

Exercise 5 fits a group's grid to the group's rows as they are when its first row is reached, after the compensation of
every earlier row. The authors' reference code has a second option, `static_groups`, that fits every grid to the original
weights before the loop starts. The next cell measures the difference on the six matrices at 3 bits, with the same
Hessians and the same dampening: the first GPTQ column is your `gptq_quantize`; the second is the slow elimination of the
check above with its switch set to `"original"`. It also counts the weights that the compensation pushed past the end of
their grid, where they were clipped (counted by the slow elimination in both settings).

In [ ]:
def _show_grid_fit() -> None:
    calib = calibration_set(POOL, CALIB_TOKENS, 0)
    bits = 3
    rows, err, clipped, total = [], {"rtn": [], "current": [], "original": []}, {"current": 0, "original": 0}, 0
    for name in QUANTISED:
        W, H = PARAMS[name], layer_hessian(calib[name])
        assert np.all(np.diag(H) > 0), f"{name} has a dead input channel, which the slow elimination does not handle"
        Hinv = np.linalg.inv(H + PERCDAMP * np.mean(np.diag(H)) * np.eye(len(H)))
        stats = {"current": {}, "original": {}}
        slow = {fit: gptq_explicit(W, Hinv, bits, GROUP_SIZE, fit, stats[fit]) for fit in stats}
        found = {"rtn": rtn_quantize(W, bits, GROUP_SIZE), "current": gptq_quantize(W, H, bits, GROUP_SIZE), "original": slow["original"]}
        for key, w_hat in found.items():
            err[key].append(rel_output_error(HELD[name], W, w_hat))
        for fit in clipped:
            clipped[fit] += stats[fit].get("clipped", 0)
        total += W.size
        rows.append([name, _percent(err["rtn"][-1]), _percent(err["current"][-1]), _percent(err["original"][-1]),
                     str(stats["current"].get("clipped", 0)), str(stats["original"].get("clipped", 0))])
    print(f"{bits}-bit, calibrated on {CALIB_TOKENS} tokens, output error on held-out text; clipped weights of {total:,} in all")
    _print_table(["matrix", "RTN", "GPTQ, grids from the current rows", "GPTQ, grids from the original weights",
                  "clipped (current rows)", "clipped (original weights)"], rows)
    mean = {k: float(np.mean(v)) for k, v in err.items()}
    print(f"\nmean over the six matrices: RTN {_percent(mean['rtn'])}, grids from the current rows {_percent(mean['current'])}, "
          f"grids from the original weights {_percent(mean['original'])}; the original-weights error is {mean['original'] / mean['current']:.2f} "
          f"times the current-rows error, with {clipped['original']} weights clipped against {clipped['current']}")
    if mean["current"] < mean["original"] and clipped["current"] < clipped["original"]:
        print("on these matrices fitting each grid to the current rows gives both the lower error and fewer clipped weights, which is "
              "what exercise 5 asks for; the compensation moves a group's weights before its grid is fitted, and a grid fitted earlier is "
              "the wrong size for them")
    else:
        print("on these matrices fitting each grid to the current rows did not give both a lower error and fewer clipped weights; "
              "the rule of exercise 5 is a choice, and the comparison above is its measured price here")


if __name__ == "__main__":
    _try("GPTQ grid fitting", _show_grid_fit, needs=("exercise 2", "exercise 3", "exercise 4", "exercise 5"))

## 7. AWQ: protect the channels that carry large activations

GPTQ uses the *correlations* between input channels. AWQ uses something simpler: how *large* each channel's
activations are. Take one weight `w` of input channel `j`, multiplied by activation `x`. Quantised, it adds an
output error of about `Δ · RoundErr · x`, where `Δ` is its group's step and `RoundErr` the rounding error
measured in steps (about the same for every weight). A channel with large `x` multiplies its rounding error by a
large number. Now scale the weight up by `s > 1` and the activation down by `s`: in full precision nothing changed.
The scaled weight is rounded on a grid with step `Δ'`, and the error becomes `Δ' · RoundErr · x / s`. The ratio of
new error to old is `(Δ' / Δ) · (1 / s)`, which is below 1 as long as `Δ' / Δ < s`. While the scaled weight stays
inside its group's range the step does not change and the ratio is `1 / s`; once it sets the range, `Δ'` grows with `s`
and the gain shrinks.

Scaling a channel up shrinks its own error and, once it sets the group's range, inflates its neighbours'. The
next cell measures this on the input channel of `Wf2` with the largest activations, using your `rtn_quantize`.
For each `s` it prints the average `Δ'/Δ` over the group's output columns, the ratio the formula predicts (squared,
because the cell measures squared error), the ratio actually measured on that channel's share of the output error,
how often the scaled weight ends its group's range, and the error of the whole layer.

In [ ]:
def _show_scaling() -> None:
    calib = calibration_set(POOL, CALIB_TOKENS, 0)
    name, bits = "Wf2", 3
    X, W = calib[name], PARAMS[name]
    j = int(np.argmax(np.abs(X).mean(axis=0)))
    g0 = (j // GROUP_SIZE) * GROUP_SIZE
    base_step = find_params(W[g0:g0 + GROUP_SIZE], bits)[0]
    contribution = lambda W_hat: float(np.mean(np.sum((X[:, [j]] * (W[[j]] - W_hat[[j]])) ** 2, axis=1)))
    base_err = contribution(rtn_quantize(W, bits, GROUP_SIZE))
    rows, layer_errs = [], []
    for s in (1.0, 1.25, 1.5, 2.0, 4.0):
        scale = np.ones(W.shape[0])
        scale[j] = s
        scaled = W * scale[:, None]
        W_hat = rtn_quantize(scaled, bits, GROUP_SIZE) / scale[:, None]
        group = scaled[g0:g0 + GROUP_SIZE]
        ratio = float(np.mean(find_params(group, bits)[0] / base_step))
        ends = float(np.mean((group[j - g0] == group.max(axis=0)) | (group[j - g0] == group.min(axis=0))))
        layer_errs.append(rel_output_error(X, W, W_hat))
        rows.append([f"{s:.2f}", f"{ratio:.3f}", f"{(ratio / s) ** 2:.3f}", f"{contribution(W_hat) / base_err:.3f}",
                     f"{100 * ends:.0f}%", _percent(layer_errs[-1])])
    print(f"{name}, channel {j} of {W.shape[0]} (the one with the largest mean |activation|), scaled alone, {bits}-bit")
    _print_table(["s", "mean Δ'/Δ", "predicted (Δ'/Δ/s)²", "measured ratio", "columns where it ends the range", "whole-layer error"], rows)
    print("\nthe prediction is squared because the measurement is an energy; one channel has just one weight per output column, so its "
          "measured ratio scatters around it with the luck of its rounding. The step Δ' grows with s because, once the "
          "scaled weight is the largest or smallest in its group, it sets the range its neighbours share")
    if min(layer_errs[1:]) < layer_errs[0]:
        print("scaling this one channel lowered the whole layer's error at some s: protecting it was worth the neighbours' loss")
    else:
        print("scaling this one channel never lowered the whole layer's error: its neighbours' loss outweighed its gain, which "
              "is why AWQ scales every channel by one smooth rule and lets the output error choose the exponent")


if __name__ == "__main__":
    _try("scaling one channel", _show_scaling, needs=("exercise 2",))

## 8. Exercise 6: the scales

AWQ does not scale one channel; it scales all of them, with one number to tune. Let `a_j` be the **mean
absolute activation** of input channel `j` on the calibration set. The scale is `s_j = a_j ^ α` for one
exponent `α` in `[0, 1]`: `α = 0` leaves everything alone (that is RTN) and `α = 1` scales in proportion to
the activations. Two conventions from the reference implementation keep the numbers sane (see `claims.yaml`):
the scales are floored at `1e-4`, applied after the power (the reference code's order), so that at `α > 0` a silent
channel with `a_j = 0` gets the scale `1e-4` and not 0, and they are divided by `sqrt(max(s) * min(s))`, which centres
them around 1.

`awq_scales(act_mag, alpha)` takes the vector `act_mag` of the `a_j` and returns the vector `s`.

<details><summary>💡 Hint 1 — what to think about</summary>

What does `0 ** 0` give in numpy, and what should a silent channel's scale be at `α = 0`? Do you floor before
or after the power, and what value does a silent channel get at `α > 0` in each case? What would dividing by the
geometric mean of the largest and smallest scale do to the pair of them?
</details>

<details><summary>💡 Hint 2 — the approach in words</summary>

Raise the magnitudes to the power `α`, floor the result at the small constant, then divide the whole vector
by the square root of its maximum times its minimum. Return a new array.
</details>

In [ ]:
def awq_scales(act_mag: np.ndarray, alpha: float) -> np.ndarray:
    """AWQ's per-input-channel scales from the mean absolute activations `act_mag` (shape `(d_in,)`).

    `s = act_mag ** alpha`, floored at 1e-4 (after the power: at `alpha > 0` a silent channel's scale is 1e-4), then
    divided by `sqrt(s.max() * s.min())`. Returns a new float array of shape `(d_in,)`; with `alpha = 0` every entry is 1.

    Worked example: act_mag [16, 4, 1] and alpha 0.5 give the powers [4, 2, 1]; sqrt(4 * 1) = 2, so
    the scales are [2, 1, 0.5].
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement awq_scales")


def _check_scales() -> None:
    got = awq_scales(np.array([16.0, 4.0, 1.0]), 0.5)
    assert np.allclose(got, [2.0, 1.0, 0.5], rtol=0, atol=1e-12), (
        f"awq_scales on the docstring example gave {np.round(got, 4).tolist()}, expected [2.0, 1.0, 0.5]: power first, "
        "then floor, then divide by sqrt(max * min)")
    mag = np.array([0.0, 0.5, 2.0, 8.0])
    assert np.array_equal(awq_scales(mag, 0.0), np.ones(4)), "alpha = 0 means no scaling: every scale is exactly 1 (0 ** 0 is 1 in numpy)"
    s = awq_scales(mag, 0.5)
    assert np.all(np.isfinite(s)) and np.all(s > 0), "a silent channel (activation 0) must get a small positive scale, not 0 or NaN: floor the scales"
    assert np.isclose(s.max() * s.min(), 1.0, rtol=1e-12), "after normalising, sqrt(max * min) must be 1, so max * min = 1"
    assert np.all(np.diff(s) >= 0), "larger activations must get larger scales (alpha > 0 keeps the order)"
    floored = awq_scales(np.array([1e-3, 1.0]), 2.0)
    assert np.allclose(floored, [0.01, 100.0], rtol=0, atol=1e-9), (
        f"awq_scales([1e-3, 1], 2) gave {np.round(floored, 4).tolist()}, expected [0.01, 100.0]: 1e-3 squared is 1e-6, below the floor of 1e-4, "
        "so the floor goes on the scale AFTER the power (flooring the activation first, or with another constant, gives another answer)")
    hi = awq_scales(np.array([1.0, 4.0, 9.0]), 1.0)
    assert np.allclose(hi / hi[0], [1.0, 4.0, 9.0], rtol=0, atol=1e-12), "alpha = 1 scales in proportion to the activations"
    print("awq_scales: the docstring example, alpha = 0, silent channels, normalisation, order, alpha = 1")


if __name__ == "__main__":
    _try("exercise 6", _check_scales)

## 9. Exercise 7: searching for alpha

The scale vector is useful only through the quantisation it feeds. For each candidate `α` in the grid
`α_j = j / n_grid`, `j = 0, 1, ..., n_grid - 1` (so `1.0` itself is not tried, as in the reference
implementation), do this:

1. `s = awq_scales(mean |X| per column, α)`;
2. quantise the scaled matrix: `rtn_quantize(W * s[:, None], bits, group_size)` (row `j` of `W` times `s_j`);
3. undo the scaling to get the **effective weights** `W_eff = rtn_quantize(...) / s[:, None]`;
4. score it by `output_error(X, W, W_eff)`, the calibration output error.

Keep the `α` with the lowest score; on a tie, the **smaller** `α` (the first one tried). Build each candidate as
`j / n_grid` from the integer `j`, not by adding `1 / n_grid` over and over, which drifts and can try an extra `α` just below
1; and compare scores with a plain `<`, no tolerance, because a score is a squared error, so a layer with small activations has
small scores that a tolerance would call equal (section 13 prints both facts). Return `(W_eff, alpha, s)` for the winner. Step 3 is the whole trick: a deployed model folds `1/s` into the operation
that produced `X` (a normalisation gain, or the previous linear layer, whose output channels it divides), and
the nanolm has no normalisation layers, so here `X @ W_eff` stands for what the folded model computes.
Because `α = 0` is a candidate, AWQ can never score worse than RTN on its own calibration set.

<details><summary>💡 Hint 1 — what to think about</summary>

Which array's column means give `act_mag`: the weights' or the calibration inputs'? Which operation scales a
whole ROW of `W` by a vector, and which divides it back? What must you keep from the best candidate, and
what is the right comparison for "a tie goes to the smaller alpha"?
</details>

<details><summary>💡 Hint 2 — the approach in words</summary>

Compute `act_mag` once. Loop over `j` in `range(n_grid)`; for each, set `α = j / n_grid`, build the scales, quantise the
scaled matrix with your `rtn_quantize`, divide back, and score. Update the best only when the score is strictly lower
than the best so far, so the first (smallest) `α` wins ties.
</details>

In [ ]:
def awq_quantize(X: np.ndarray, W: np.ndarray, bits: int, group_size: int, n_grid: int = N_GRID):
    """AWQ for one matrix: the effective weights of the best `alpha` on the grid `j / n_grid`.

    `X` is `(n, d_in)` calibration inputs, `W` is `(d_in, d_out)`. For each candidate: scales from
    `awq_scales(np.abs(X).mean(axis=0), alpha)`, `W_eff = rtn_quantize(W * s[:, None], bits, group_size) /
    s[:, None]`, score `output_error(X, W, W_eff)`. Each alpha is `j / n_grid` for the integer `j`. The candidate with the
    strictly lowest score wins (a plain `<`, no tolerance); ties go to the smaller alpha. Returns `(W_eff, alpha, s)`; `X`
    and `W` are not changed.

    Worked example: with n_grid 1 the only candidate is alpha 0, whose scales are all 1, so the answer is
    `rtn_quantize(W, bits, group_size)`; so it is when X is all zeros and every candidate scores 0.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement awq_quantize")


def _check_awq() -> None:
    rng = np.random.default_rng(41)
    W = rng.normal(0, 1, (12, 5))
    X = rng.normal(0, 1, (60, 12)) * np.exp(1.2 * rng.normal(0, 1, 12))
    W_in, X_in = W.copy(), X.copy()
    W_eff, alpha, s = awq_quantize(X, W, 3, 4)
    assert np.array_equal(W, W_in) and np.array_equal(X, X_in), "awq_quantize changed its inputs: work on new arrays"
    assert W_eff.shape == W.shape and s.shape == (12,), "W_eff has the shape of W and s has one entry per input channel"
    assert 0.0 <= alpha < 1.0 and abs(alpha * N_GRID - round(alpha * N_GRID)) < 1e-9, (
        f"alpha {alpha} is not one of j / n_grid for j in 0 .. n_grid - 1")
    assert np.allclose(s, awq_scales(np.abs(X).mean(axis=0), alpha), rtol=0, atol=1e-12), "the returned s must be the scales of the winning alpha"
    assert np.allclose(W_eff, rtn_quantize(W * s[:, None], 3, 4) / s[:, None], rtol=0, atol=1e-12), (
        "W_eff must be the quantised SCALED matrix divided back by s (row j of W times s_j, then row j of the result divided by s_j)")
    plain = rtn_quantize(W, 3, 4)
    assert output_error(X, W, W_eff) <= output_error(X, W, plain) + 1e-12, (
        "alpha = 0 is a candidate, so AWQ must score at least as well as round-to-nearest on its own calibration set")
    assert alpha > 0.0, "with activations this uneven the best alpha should not be 0: score every candidate and keep the lowest"
    scores = [output_error(X, W, rtn_quantize(W * awq_scales(np.abs(X).mean(axis=0), j / N_GRID)[:, None], 3, 4)
                           / awq_scales(np.abs(X).mean(axis=0), j / N_GRID)[:, None]) for j in range(N_GRID)]
    assert output_error(X, W, W_eff) <= min(scores) + 1e-12, (
        "some alpha on the grid scores better than the one you returned: score every candidate with output_error(X, W, W_eff), "
        "the error of the layer's OUTPUT on the calibration inputs (not of the weights), and keep the lowest")
    # channels that are both uneven and correlated: scoring the weights, an activation-weighted weight error, or the error of the
    # scaled inputs X / s picks another alpha on data like this, so the winner must be the brute-force lowest output error
    for seed in (4, 7):
        gen = np.random.default_rng(seed)
        Wb = gen.normal(0, 1, (12, 5))
        Xb = (gen.normal(0, 1, (60, 12)) @ (np.eye(12) + 0.6 * gen.normal(0, 1, (12, 12)))) * np.exp(1.2 * gen.normal(0, 1, 12))
        mag = np.abs(Xb).mean(axis=0)
        errs = [output_error(Xb, Wb, rtn_quantize(Wb * awq_scales(mag, j / N_GRID)[:, None], 3, 4) / awq_scales(mag, j / N_GRID)[:, None])
                for j in range(N_GRID)]
        best = int(np.argmin(errs))
        _, alpha_b, _ = awq_quantize(Xb, Wb, 3, 4)
        assert abs(alpha_b - best / N_GRID) < 1e-9, (
            f"on uneven, correlated channels (data set {seed}) your alpha is {alpha_b:.2f} but the lowest calibration output error is at "
            f"{best / N_GRID:.2f}. Score each candidate with output_error(X, W, W_eff), the output error on the calibration inputs against "
            "the ORIGINAL weights: a weight error, an activation-weighted weight error or the error of the scaled inputs X / s choose "
            "another alpha here")
    # two float traps. A grid built by adding 1 / n_grid again and again drifts: for n_grid = 10 it tries an extra alpha just below 1.
    # And a tie rule with a tolerance swallows real differences when the activations are small: a score is a squared error.
    def lowest(Xs, Ws, bits, group, n_grid):
        mags = np.abs(Xs).mean(axis=0)
        errs = [output_error(Xs, Ws, rtn_quantize(Ws * awq_scales(mags, j / n_grid)[:, None], bits, group) / awq_scales(mags, j / n_grid)[:, None])
                for j in range(n_grid)]
        return int(np.argmin(errs)) / n_grid
    loud = np.random.default_rng(0).normal(0, 1, (80, 16)) * np.array([1.0] * 15 + [100.0])      # one channel 100 times louder
    W_loud = np.random.default_rng(50).normal(0, 1, (16, 4))
    _, alpha_loud, _ = awq_quantize(loud, W_loud, 2, 4, n_grid=10)
    assert abs(alpha_loud - lowest(loud, W_loud, 2, 4, 10)) < 1e-12 and alpha_loud < 0.95, (
        f"with n_grid = 10 the candidates are 0, 0.1, ..., 0.9, and you returned {alpha_loud!r}. Build each alpha as j / n_grid from an "
        "integer j: adding 1 / n_grid over and over drifts, and the sum just below 1 becomes a tenth candidate")
    for factor in (1e-4, 1e-6):
        X_small = X * factor
        _, alpha_small, _ = awq_quantize(X_small, W, 3, 4)
        assert abs(alpha_small - lowest(X_small, W, 3, 4, N_GRID)) < 1e-12, (
            f"with the activations scaled by {factor:g} the lowest score is at alpha {lowest(X_small, W, 3, 4, N_GRID):.2f} and you returned "
            f"{alpha_small:.2f}. Compare with a plain <, no tolerance: a score is a squared error, so small activations give small scores "
            "that a tolerance, absolute or np.isclose, calls equal")
    one, a1, s1 = awq_quantize(X, W, 3, 4, n_grid=1)
    assert a1 == 0.0 and np.array_equal(one, plain) and np.array_equal(s1, np.ones(12)), "with n_grid = 1 the only candidate is alpha 0, which is plain round-to-nearest"
    _, a0, _ = awq_quantize(np.zeros((5, 12)), W, 3, 4)
    assert a0 == 0.0, "when every candidate scores the same, the FIRST (smallest alpha) must win: compare with <, not <="
    print("awq_quantize: the winning alpha, effective weights, never worse than RTN, the end of the grid, small scores, n_grid = 1, ties")


if __name__ == "__main__":
    _try("exercise 7", _check_awq, needs=("exercise 2", "exercise 6"))

### Where AWQ helped in this model

For each matrix: the **activation spread** (the largest channel's mean absolute activation divided by the median
channel's: 1 would mean all channels look alike), the `α` AWQ chose, and the held-out output errors of RTN and
AWQ at 3 bits. If AWQ's reasoning is right, the gain should tend to follow the spread. That is a weak test: the
matrices that read the same input share one spread (the cell counts the distinct ones), and other things move the
gain too. The synthetic layers of section 10 are the controlled test.

In [ ]:
def _show_awq_layers() -> None:
    calib = calibration_set(POOL, CALIB_TOKENS, 0)
    rows, spreads, savings = [], [], []
    for name in QUANTISED:
        W, X = PARAMS[name], calib[name]
        mag = np.abs(X).mean(axis=0)
        W_awq, alpha, _ = awq_quantize(X, W, 3, GROUP_SIZE)
        e_rtn = rel_output_error(HELD[name], W, rtn_quantize(W, 3, GROUP_SIZE))
        e_awq = rel_output_error(HELD[name], W, W_awq)
        spreads.append(float(mag.max() / np.median(mag)))
        savings.append(1 - e_awq / e_rtn)
        rows.append([name, f"{spreads[-1]:.2f}", f"{alpha:.2f}", _percent(e_rtn), _percent(e_awq), f"{100 * savings[-1]:.1f}%"])
    print("3-bit quantisation, AWQ calibrated on", CALIB_TOKENS, "tokens, errors on held-out text")
    _print_table(["matrix", "spread", "alpha", "RTN error", "AWQ error", "AWQ saves"], rows)
    pairs = [(a, b) for a in range(len(rows)) for b in range(a + 1, len(rows)) if spreads[a] != spreads[b] and savings[a] != savings[b]]
    agree = sum((spreads[a] - spreads[b]) * (savings[a] - savings[b]) > 0 for a, b in pairs)
    print(f"\nof the {len(pairs)} pairs of matrices whose spread differs, {agree} are ordered the same way by spread and by AWQ's saving; "
          f"a weak test, since the {len(rows)} matrices have only {len(set(spreads))} distinct spreads (matrices that read one input share it)")


if __name__ == "__main__":
    _try("AWQ per matrix", _show_awq_layers, needs=("exercise 2", "exercise 3", "exercise 7"))

## 10. Which structure does each method use?

GPTQ's compensation is worth something only if input channels are *correlated* (the ratio from exercise 4
is 1 otherwise; the synthetic inputs below have zero mean, so there the two statements say the same); AWQ's scaling
is aimed at channels whose *magnitudes* differ, and should do most where they do. A real model
has some of both, in amounts you cannot dial. A synthetic layer lets you dial each. The cell below builds
layers (`make_synthetic_layer`) whose inputs have neighbour correlation `rho` (channel `i` and `j` correlate as
`rho ** |i - j|`) and log-normal channel magnitudes with spread `spread` (0 gives equal magnitudes), draws a
calibration set and a test set from the same distribution, quantises to 3 bits with all three methods, and
prints each method's test error **as a fraction of RTN's** (below 1 is better than RTN). Same seed, same weights
and same underlying noise in every cell: only `rho` and `spread` change.

In [ ]:
def make_synthetic_layer(rho: float, spread: float, d_in: int = 64, d_out: int = 64, n: int = 1024, seed: int = SYNTH_SEED):
    """Returns (calibration inputs, test inputs, weights) for a layer with the given channel structure."""
    rng = np.random.default_rng(seed)
    idx = np.arange(d_in)
    chol = np.linalg.cholesky(rho ** np.abs(idx[:, None] - idx[None, :]))
    magnitude = np.exp(spread * rng.normal(0, 1, d_in))
    calibration = (rng.normal(0, 1, (n, d_in)) @ chol.T) * magnitude
    test = (rng.normal(0, 1, (n, d_in)) @ chol.T) * magnitude
    return calibration, test, rng.normal(0, 1, (d_in, d_out)) / np.sqrt(d_in)


RHOS, SPREADS = (0.0, 0.5, 0.9), (0.0, 0.75, 1.5)


def _show_regimes() -> None:
    bits = 3
    rows, ratio, scale_range = [], {}, {}
    for spread in SPREADS:
        row = [f"spread {spread:.2f}"]
        for rho in RHOS:
            cal, test, W = make_synthetic_layer(rho, spread)
            e_rtn = rel_output_error(test, W, rtn_quantize(W, bits, GROUP_SIZE))
            W_awq, _, s_awq = awq_quantize(cal, W, bits, GROUP_SIZE)
            scale_range[(spread, rho)] = float(s_awq.max() / s_awq.min())
            e_awq = rel_output_error(test, W, W_awq)
            e_gptq = rel_output_error(test, W, gptq_quantize(W, layer_hessian(cal), bits, GROUP_SIZE))
            ratio[(spread, rho)] = (e_awq / e_rtn, e_gptq / e_rtn)
            row.append(f"AWQ {e_awq / e_rtn:.2f}  GPTQ {e_gptq / e_rtn:.2f}")
        rows.append(row)
    print(f"{bits}-bit test error as a fraction of RTN's (below 1 beats RTN); rows differ in channel magnitudes, columns in correlation")
    _print_table(["", *[f"rho {r:.1f}" for r in RHOS]], rows)
    lo_s, hi_s, lo_r, hi_r = SPREADS[0], SPREADS[-1], RHOS[0], RHOS[-1]
    cal0, test0, _ = make_synthetic_layer(lo_r, lo_s)
    corr = np.corrcoef(cal0, rowvar=False)
    np.fill_diagonal(corr, 0.0)
    i0, j0 = np.unravel_index(np.argmax(np.abs(corr)), corr.shape)
    print(f"at rho {lo_r:.1f} the channels are uncorrelated by construction, yet the {len(cal0):,} calibration tokens show a sample "
          f"correlation of {round(corr[i0, j0], 2) + 0.0:.2f} between channels {i0} and {j0} (the same pair on the test tokens: "
          f"{round(np.corrcoef(test0, rowvar=False)[i0, j0], 2) + 0.0:.2f}); chance patterns like that are what GPTQ has to compensate along there"
          + (f", and its test error at rho {lo_r:.1f} is above RTN's for every spread" if min(ratio[(sp, lo_r)][1] for sp in SPREADS) > 1.0 else ""))
    print(f"\nwith uncorrelated channels, AWQ's error goes from {ratio[(lo_s, lo_r)][0]:.2f} to {ratio[(hi_s, lo_r)][0]:.2f} of RTN's as the "
          f"spread grows, while GPTQ's goes from {ratio[(lo_s, lo_r)][1]:.2f} to {ratio[(hi_s, lo_r)][1]:.2f}")
    print(f"with equal magnitudes, GPTQ's error goes from {ratio[(lo_s, lo_r)][1]:.2f} to {ratio[(lo_s, hi_r)][1]:.2f} of RTN's as the "
          f"correlation grows, while AWQ's goes from {ratio[(lo_s, lo_r)][0]:.2f} to {ratio[(lo_s, hi_r)][0]:.2f}")
    print(f"at equal magnitudes and the highest correlation the scales AWQ chose differ by a factor of {scale_range[(lo_s, hi_r)]:.2f} "
          f"between the largest and the smallest (at the highest spread: {scale_range[(hi_s, hi_r)]:.2f})"
          + (f"; scales that close to 1 leave the unequal-magnitudes mechanism little to work with, so the gain that remains there comes from "
             f"the search itself: the best of {N_GRID} candidate scalings, chosen on the calibration set, re-rounds the weights a little "
             "differently from RTN" if scale_range[(lo_s, hi_r)] < 2.0 else ""))


if __name__ == "__main__":
    _try("regime map", _show_regimes, needs=("exercise 2", "exercise 3", "exercise 5", "exercise 7"))

## 11. The whole model, at 4 and 3 bits

Now the comparison the lesson is for. The three methods quantise the six matrices from one calibration set
of `CALIB_TOKENS` tokens, and the quantised matrices are put back into the model. AWQ and GPTQ are run on three
independent calibration draws, because their result depends on which tokens they saw. The cell reports, for each
method and bit width: the mean held-out output error of the six matrices, the **KL divergence** from the
full-precision model (mean, and the range over the draws), and the **held-out loss change** with a standard
error over the held-out windows. The model is small and has partly memorised its text, so differences of a few
hundredths in loss are not all signal; KL, which is never negative, is the steadier yardstick here.

In [ ]:
METHODS = ("rtn", "awq", "gptq")


def quantize_matrix(method: str, X: np.ndarray, W: np.ndarray, bits: int, group_size: int = GROUP_SIZE) -> np.ndarray:
    """One matrix by one method, calibrated on `X`: the effective weights. Given; it calls your functions."""
    if method == "rtn":
        return rtn_quantize(W, bits, group_size)
    if method == "gptq":
        return gptq_quantize(W, layer_hessian(X), bits, group_size, PERCDAMP)
    if method == "awq":
        return awq_quantize(X, W, bits, group_size, N_GRID)[0]
    raise ValueError(f"unknown method {method!r}")


def quantize_model(method: str, bits: int, calib: dict) -> dict:
    return {name: quantize_matrix(method, calib[name], PARAMS[name], bits) for name in QUANTISED}


def window_losses(params: dict) -> np.ndarray:
    """Mean cross-entropy of each held-out window, shape (n_windows,)."""
    lp = log_probs(params, VAL_X)
    return -np.take_along_axis(lp, VAL_Y[..., None], axis=-1)[..., 0].mean(axis=1)


def evaluate_model(replacements: dict) -> dict:
    """Mean held-out matrix error, KL to full precision, and the per-window held-out loss change."""
    params = with_weights(PARAMS, replacements)
    return {"layer": float(np.mean([rel_output_error(HELD[n], PARAMS[n], replacements[n]) for n in QUANTISED])),
            "kl": kl_to_full_precision(PARAMS, params, VAL_X), "dloss": window_losses(params) - window_losses(PARAMS)}


def evaluate_method(method: str, bits: int) -> dict:
    """evaluate_model over the independent calibration draws (RTN looks at no data, so one draw): the
    mean matrix error, the mean and the range of the KL, and the loss change averaged over draws, with
    its standard error over the held-out windows."""
    runs = [evaluate_model(quantize_model(method, bits, calibration_set(POOL, CALIB_TOKENS, d)))
            for d in (CALIB_DRAWS if method != "rtn" else CALIB_DRAWS[:1])]
    diff = np.mean([r["dloss"] for r in runs], axis=0)
    kls = [r["kl"] for r in runs]
    return {"layer": float(np.mean([r["layer"] for r in runs])), "kl": float(np.mean(kls)), "kl_lo": min(kls), "kl_hi": max(kls),
            "dloss": float(diff.mean()), "se": float(diff.std(ddof=1) / np.sqrt(len(diff)))}


def _compare_models() -> None:
    results = {(m, b): evaluate_method(m, b) for b in BITS for m in METHODS}
    print(f"calibration sets of {CALIB_TOKENS} tokens, {len(CALIB_DRAWS)} independent draws for AWQ and GPTQ; "
          f"full-precision held-out loss {BASELINE['loss']:.3f} nats per character")
    rows = [[f"{m.upper()} {b}-bit", _percent(r["layer"]), f"{r['kl']:.4f}",
             "" if m == "rtn" else f"{r['kl_lo']:.4f} to {r['kl_hi']:.4f}", f"{r['dloss']:+.3f} ± {r['se']:.3f}"]
            for (m, b), r in ((k, results[k]) for k in sorted(results, key=lambda k: (-k[1], METHODS.index(k[0]))))]
    _print_table(["method", "mean matrix error", "KL (nats)", "KL over the draws", "held-out loss change"], rows)
    for b in BITS:
        order = sorted(METHODS, key=lambda m: results[(m, b)]["kl"])
        print(f"at {b} bits, by mean KL: " + ", ".join(m.upper() for m in order) + " (smallest first)")
        for lo_m, hi_m in zip(order, order[1:]):
            clear = results[(lo_m, b)]["kl_hi"] < results[(hi_m, b)]["kl_lo"]
            print(f"  {lo_m.upper()} is clearly below {hi_m.upper()}" if clear else
                  f"  {lo_m.upper()} and {hi_m.upper()} are not separated: their KL ranges over the draws overlap")
    better = [f"{m.upper()} {b}-bit" for (m, b), r in results.items() if r["dloss"] < 0]
    if better:
        print("held-out loss went DOWN for " + ", ".join(better) + ": the quantised model scored better than the full-precision "
              "one on this text, a sign that loss differences this small are not a reliable quality signal here")
    for b in BITS:
        a, r = results[("awq", b)], results[("rtn", b)]
        if a["layer"] < r["layer"] and a["kl_hi"] > r["kl"]:
            print(f"at {b} bits AWQ lowers the matrix error but its KL reaches above RTN's in at least one draw: matrix-by-matrix "
                  "error is what AWQ minimises, and it is not the end-to-end quantity")


if __name__ == "__main__":
    _try("whole-model comparison", _compare_models, needs=("exercise 2", "exercise 3", "exercise 5", "exercise 7"))

### The same comparison, matrix by matrix

The mean hides which matrices are hard. Here are the held-out output errors of each matrix for the three
methods at both bit widths, from the same calibration set.

In [ ]:
def _compare_matrices() -> None:
    calib = calibration_set(POOL, CALIB_TOKENS, 0)
    errs = {(m, b): {n: rel_output_error(HELD[n], PARAMS[n], w) for n, w in quantize_model(m, b, calib).items()}
            for b in BITS for m in METHODS}
    header = ["matrix", *[f"{m.upper()} {b}-bit" for b in BITS for m in METHODS]]
    _print_table(header, [[n, *[_percent(errs[(m, b)][n]) for b in BITS for m in METHODS]] for n in QUANTISED])


if __name__ == "__main__":
    _try("matrix by matrix", _compare_matrices, needs=("exercise 2", "exercise 3", "exercise 5", "exercise 7"))

## 12. Exercise 8: how much calibration data is enough?

GPTQ builds a `d_in × d_in` Hessian out of the calibration tokens. With fewer tokens than input channels that
matrix has no inverse; dampening forces one, but the compensation then fits the few tokens it saw, and held-out
text pays. AWQ only measures `d_in` average magnitudes and tunes one exponent, so it should need far less. The
AWQ paper reports exactly this (see `claims.yaml`); the sweep below measures it on this model, for sets from
the smallest to the largest size in `SWEEP_TOKENS`, with `CALIB_DRAWS` independent draws each.

"Stops improving" needs a rule. Use this one: given sizes in increasing order and their mean errors, the
**knee** is the smallest size from which *every* larger size, itself included, has an error of at most
`errors[-1] * (1 + tol)`, the error of the largest set plus a margin. Requiring *every* larger size matters:
a noisy curve can dip into the band, climb out again, and has not stopped improving. The comparison is the
floating-point `error <= errors[-1] * (1 + tol)`, and the checks use only numbers that are exact in binary.

`knee_size(sizes, errors, tol)` returns that size (an element of `sizes`); `sizes` and `errors` may each be a list, a
tuple or a numpy array. It raises `ValueError` when they are empty or of different lengths, when `sizes` is not strictly
increasing, or when `tol` is negative.

<details><summary>💡 Hint 1 — what to think about</summary>

Which entry decides the threshold? For a candidate position, what has to be true of everything after it?
Which position would you try first if you wanted the *smallest* size, and which would you try last?
</details>

<details><summary>💡 Hint 2 — the approach in words</summary>

Compute the threshold once. Walk backwards from the end while the errors stay at or below it; the first
position where one is above, plus one, is the knee. Validate the inputs before any of that.
</details>

In [ ]:
def knee_size(sizes, errors, tol: float):
    """The smallest size from which every larger size has `error <= errors[-1] * (1 + tol)`.

    `sizes` and `errors` may each be a list, a tuple or a numpy array. Raises ValueError if they are empty or
    differ in length, if `sizes` is not strictly increasing, or if `tol < 0`. The last size always qualifies.

    Worked example: sizes [10, 20, 30, 40], errors [4.0, 1.5, 1.0, 1.0], tol 0.5: the threshold is 1.5,
    and from size 20 on every error is at most 1.5, so the answer is 20. With sizes [1, 2, 3, 4, 5],
    errors [5.0, 1.0, 3.0, 1.0, 1.0] and tol 0.25 the answer is 4: size 2 is inside the band but size 3 is not.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement knee_size")


def _check_knee() -> None:
    got = knee_size([10, 20, 30, 40], [4.0, 1.5, 1.0, 1.0], 0.5)
    assert got == 20, f"knee_size on the first docstring example gave {got}, expected 20: the band is errors[-1] * (1 + tol), inclusive"
    got = knee_size([1, 2, 3, 4, 5], [5.0, 1.0, 3.0, 1.0, 1.0], 0.25)
    assert got == 4, (
        f"knee_size on the second docstring example gave {got}, expected 4: size 2 is inside the band but size 3 climbs "
        "out of it, so it has not stopped improving. The knee needs EVERY larger size inside the band")
    assert knee_size([8, 16], [2.0, 1.0], 0.0) == 16 and knee_size([8, 16], [1.0, 1.0], 0.0) == 8, "tol = 0: equal to the last error counts as inside"
    assert knee_size([1, 2, 3, 4], [10.0, 6.0, 5.0, 4.0], 0.5) == 2, "the band is errors[-1] * (1 + tol) = 6, a MULTIPLICATIVE margin, not errors[-1] + tol"
    assert knee_size([1, 2, 3], [3.0, 1.0, 2.0], 0.5) == 1, "the threshold comes from the LAST error, not the smallest: an earlier dip below it does not move it"
    assert knee_size([1, 2, 3], [1.5, 1.0, 1.0], 0.5) == 1 and knee_size([1, 2, 3], [1.5000000000000002, 1.0, 1.0], 0.5) == 2, (
        "compare exactly with <=: an error equal to the threshold is inside, one unit in the last place above it is outside")
    assert knee_size(np.array([16, 32, 64]), np.array([4.0, 1.0, 1.0]), 0.5) == 32 and knee_size((3, 5), (1.0, 1.0), 0.1) == 3, (
        "sizes and errors may be tuples or numpy arrays as well as lists: convert them with list() before you test whether they are empty")
    assert knee_size([5], [3.0], 0.1) == 5, "a single size is its own knee"
    assert knee_size([1, 2, 3], [1.0, 1.0, 1.0], 0.0) == 1, "a flat curve stopped improving at the first size"
    for bad in (([], [], 0.1), ([1, 2], [1.0], 0.1), ([2, 1], [1.0, 1.0], 0.1), ([1, 1], [1.0, 1.0], 0.1), ([1, 2], [1.0, 1.0], -0.1)):
        try:
            knee_size(*bad)
        except ValueError:
            continue
        raise AssertionError(f"knee_size{bad} should raise ValueError")
    print("knee_size: the docstring examples, the inclusive band, every-larger-size, single and flat curves, bad inputs")


if __name__ == "__main__":
    _try("exercise 8", _check_knee)

### The sweep

For each calibration size the cell quantises all six matrices to 3 bits with AWQ and with GPTQ, once per
independent draw, and averages. It prints the error on held-out text, the error on the calibration tokens
themselves, and their ratio, which is how much the method gained from having seen those exact tokens. RTN
does not look at the data, so it is one number. Then it applies your `knee_size`.

In [ ]:
def calibration_sweep(bits: int = 3) -> dict:
    """Mean over the six matrices and the draws: {method: {"held": [...], "calib": [...]}} per size in SWEEP_TOKENS."""
    out = {m: {"held": [], "calib": []} for m in ("awq", "gptq")}
    for n in SWEEP_TOKENS:
        for m in out:
            held, own = [], []
            for draw in CALIB_DRAWS:
                calib = calibration_set(POOL, n, draw)
                w_hat = {name: quantize_matrix(m, calib[name], PARAMS[name], bits) for name in QUANTISED}
                held.append(np.mean([rel_output_error(HELD[k], PARAMS[k], w_hat[k]) for k in QUANTISED]))
                own.append(np.mean([rel_output_error(calib[k], PARAMS[k], w_hat[k]) for k in QUANTISED]))
            out[m]["held"].append(float(np.mean(held)))
            out[m]["calib"].append(float(np.mean(own)))
    return out


def _show_sweep() -> None:
    bits = 3
    sweep = calibration_sweep(bits)
    rtn_err = float(np.mean([rel_output_error(HELD[n], PARAMS[n], rtn_quantize(PARAMS[n], bits, GROUP_SIZE)) for n in QUANTISED]))
    widest = max(PARAMS[n].shape[0] for n in QUANTISED)
    rows = []
    for k, n in enumerate(SWEEP_TOKENS):
        a, g = sweep["awq"], sweep["gptq"]
        rows.append([str(n), f"{n / widest:.2f}", _percent(a["held"][k]), _percent(a["calib"][k]), _percent(g["held"][k]),
                     _percent(g["calib"][k]), f"{g['held'][k] / g['calib'][k]:.2f}"])
    print(f"{bits}-bit, mean over the six matrices and {len(CALIB_DRAWS)} draws; RTN's held-out error is {_percent(rtn_err)} at every size")
    _print_table(["tokens", "tokens / widest d_in", "AWQ held-out", "AWQ own tokens", "GPTQ held-out", "GPTQ own tokens",
                  "GPTQ held/own"], rows)
    knees = {m: knee_size(SWEEP_TOKENS, sweep[m]["held"], KNEE_TOL) for m in ("awq", "gptq")}
    print(f"\nknee (within {100 * KNEE_TOL:.0f}% of the error at {SWEEP_TOKENS[-1]} tokens, from there on): "
          f"AWQ {knees['awq']} tokens, GPTQ {knees['gptq']} tokens; the widest matrix has {widest} input channels")
    first = sweep["gptq"]["held"][0]
    print(f"with {SWEEP_TOKENS[0]} tokens GPTQ's held-out error is {_percent(first)}, "
          f"{'above' if first > rtn_err else 'below'} RTN's {_percent(rtn_err)}")


if __name__ == "__main__":
    _try("calibration sweep", _show_sweep, needs=("exercise 2", "exercise 3", "exercise 5", "exercise 7", "exercise 8"))

## 13. Common mistakes

- **Judging a quantiser by its weight error.** Round-to-nearest wins that contest by construction, and
  GPTQ deliberately loses it. The error that matters is the one in the layer's output, on text the
  calibration set did not contain.
- **Reading the wrong triangle.** `np.linalg.cholesky` returns the *lower* factor. GPTQ needs the upper one,
  `Uᵀ U = H⁻¹`; with the lower factor the entries to the right of the diagonal are zero and the method silently
  becomes round-to-nearest.
- **Fitting a group's grid to the original weights when exercise 5 asks for the compensated rows.** By the time a
  group's first row is reached, earlier rows have moved it. Fitting to the original weights is a deliberate option in
  the authors' reference code (`static_groups`), but then the compensation can push values past the ends of a grid
  fitted to the old range, where they clip (the cell after "GPTQ against RTN" counts them); the check against explicit
  elimination tests the rule the exercise states.
- **Taking the damping's mean before the dead-channel fix, or skipping the fix.** The authors' code does the fix first.
- **Scaling the weights and forgetting the activations.** AWQ's identity needs both halves: `W * s[:, None]`
  and `X / s`. Apply only one and the full-precision output changes.
- **Flooring AWQ's scales before the power, or not at all.** `0 ** alpha` is 0 for any positive `alpha`, so with no
  floor a silent channel's scale is 0 and dividing by it fails. Flooring the activation before the power keeps the
  scale positive but gives a silent channel another value; the reference code floors after the power, so exercise 6
  does too.
- **Building AWQ's alpha grid by repeated addition, or comparing scores with a tolerance.** Adding `1 / n_grid` again and
  again drifts and can try an extra candidate just below 1; a tolerance such as `np.isclose` calls small scores equal. Build
  `j / n_grid` from the integer `j` and compare with a plain `<`.
- **Trusting a calibration set smaller than the layer is wide.** The Hessian is singular; dampening hides
  that from the solver, not from the held-out text.
- **Comparing loss differences smaller than their standard error.** Use a yardstick that cannot go negative,
  such as KL, alongside the loss.

In [ ]:
# Python facts behind four of those mistakes, computed rather than asserted.
_probe = np.linalg.cholesky(np.array([[4.0, 2.0], [2.0, 3.0]]))
print(f"np.linalg.cholesky returns a lower triangle: the entry above its diagonal is {_probe[0, 1]}")
print(f"0.0 ** 0.5 is {0.0 ** 0.5}, so a silent channel's scale is 0 unless something floors it; floored after the power at 1e-4 "
      f"it is {max(0.0 ** 0.5, 1e-4)}, floored before the power it is {max(0.0, 1e-4) ** 0.5}: both are positive, "
      f"and they are not the same number; 0.0 ** 0.0 is {0.0 ** 0.0}")
_n, _alpha, _tried = 10, 0.0, []
while _alpha < 1.0:                       # a grid of _n candidates built by adding 1 / _n again and again
    _tried.append(_alpha)
    _alpha += 1 / _n
print(f"adding 1 / {_n} again and again from 0 gives {len(_tried)} values below 1.0 where {_n} candidates were meant: the last is "
      f"{_tried[-1]!r}, while {_n - 1} / {_n} is {(_n - 1) / _n!r}; build alpha as j / n_grid")
_big, _small = 3e-9, 1e-9
print(f"np.isclose({_big:g}, {_small:g}) is {np.isclose(_big, _small)}: a tolerance can call a score {_big / _small:g} times another equal "
      f"when scores are small, which is why exercise 7 compares with a plain <")

## 14. Self-check

1. After GPTQ has run, each quantised weight is:
   - (a) the grid point nearest to its original value, exactly as in round-to-nearest
   - (b) the grid point nearest to its value *after* the rounding errors of earlier channels were pushed onto it,
     so it can be farther from the original than round-to-nearest's choice would be
   - (c) always one grid step above its original value, to leave room for compensation
   - (d) the grid point nearest to the average activation of its input channel

2. With the same grids, GPTQ and round-to-nearest give identical results when:
   - (a) the model has fewer than a billion parameters
   - (b) the calibration set has more tokens than the layer has weights
   - (c) every output channel of the layer has the same weights
   - (d) the Hessian is diagonal, because then no rounding error can be compensated by moving another weight

3. AWQ multiplies the weights of input channel `j` by `s_j > 1` and divides the layer's input by `s_j`. Why does
   this protect the channel?
   - (a) The scaled weight is rounded on a grid with about the same step, so its error, measured back in
     the original units, shrinks by about `s_j`, unless the scaling made it set the range of its group
   - (b) Larger numbers are stored with more bits
   - (c) It moves the weight away from the nearest rounding boundary
   - (d) It changes the full-precision output so that the error is hidden in the difference

4. A calibration set has fewer tokens than the layer has input channels. What does GPTQ do?
   - (a) Nothing different: dampening makes the Hessian invertible, so it behaves as with a large set
   - (b) It refuses to run
   - (c) The sample Hessian is singular and dampening only repairs it for the solver: the compensation fits the
     tokens it saw, so the error on those tokens is far below the error on held-out text
   - (d) It falls back to round-to-nearest

5. The lesson reports KL divergence from the full-precision model next to held-out loss because:
   - (a) KL is much cheaper to compute
   - (b) a quantised model can score a lower loss than the original by chance, and loss differences within their
     standard error cannot separate the methods, while KL is never negative and measures how far the model moved
   - (c) loss cannot be computed for a quantised model
   - (d) KL is the quantity GPTQ minimises

<details><summary>Answers</summary>

Answers come with this lesson's worked solution when you enrol on Synapsa.
</details>

In [ ]:
print("self-check: work the questions above, then compare with the worked solution")

## 15. What this lesson does not cover

This lesson keeps everything at a scale where you can read all of it, and says where that differs from the
models people actually compress. The model is one block with a few tens of thousands of quantised weights
trained on a few thousand characters, with no normalisation layers and activations that are nearly even across
channels; a large language model has many blocks and a few input channels whose activations are far
larger than the rest, which is the situation AWQ was designed for. Each matrix
here is quantised from the full-precision model's activations, one matrix at a time; a deeper model can instead feed
each block the output of the blocks already quantised before it. The searches are the simplest that show the idea: AWQ
also applies weight clipping, and its reference code scores whole blocks, not single matrices, with one
scale shared by the matrices that read the same input; GPTQ's reference code updates columns in lazy
batches (so some group grids are fitted to slightly less compensated weights than here) and can reorder
channels by Hessian diagonal. Nothing here packs integers into memory, writes a
fast kernel, quantises activations (SmoothQuant, W8A8) or the KV cache, or handles FP8. As of 2026-10-14 the
AutoGPTQ and AutoAWQ repositories are archived and point to other tools (`claims.yaml` has the sources), so
the methods you built now live in maintained toolkits such as GPTQModel and llm-compressor.

**Versions this lesson follows,** as read on 2026-10-14: the GPTQ paper (arXiv 2210.17323, ICLR 2023, second
version of March 2023) and the AWQ paper (arXiv 2306.00978, MLSys 2024, sixth version of April 2026). Papers
and reference code change; `claims.yaml` has the quotes and the dates.

## What you built

An integer grid and round-to-nearest; the layer-wise objective as a quadratic form in the Hessian; the Optimal
Brain Surgeon update; GPTQ as one Cholesky factor and one loop, proved equal to explicit elimination; AWQ's
scales and their search; and a measurement of what each method buys on a model you trained, at 4 and 3 bits,
from how much calibration data. The habit to keep: ask which structure of the data a method uses (correlation
between channels, or their relative size), and check on data where you can switch that structure off.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now: each one already printed its feedback in its own cell above, so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in (
                ("exercise 1", _check_grid, ()),
                ("exercise 2", _check_rtn, ("exercise 1",)),
                ("exercise 3", _check_hessian, ()),
                ("exercise 4", _check_obs, ()),
                ("exercise 5", _check_gptq, ("exercise 1", "exercise 2", "exercise 4")),
                ("exercise 6", _check_scales, ()),
                ("exercise 7", _check_awq, ("exercise 2", "exercise 6")),
                ("exercise 8", _check_knee, ())):
            _try(_name, _check, needs=_needs)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.